# 1: Setup & Installationen

In [ ]:
# --- Google Drive Mount und Projektpfad ---
try:
    from google.colab import drive
    drive.mount('/content/drive')

    import os

    # !!! WICHTIG: Pfad zu deinem Projektordner in Google Drive anpassen !!!
    #             (Der Ordner, wo RPG.ipynb und die .py Dateien liegen)
    project_path = '/content/drive/MyDrive/Colab Notebooks/' # Beispielpfad, ANPASSEN!

    # Versuche, in das Projektverzeichnis zu wechseln
    os.chdir(project_path)
    print(f"Aktuelles Arbeitsverzeichnis: {os.getcwd()}")

except ImportError:
    print("Google Colab Umgebung nicht erkannt. Arbeite im aktuellen Verzeichnis.")
    project_path = os.getcwd()
    print(f"Aktuelles Arbeitsverzeichnis: {project_path}")
except FileNotFoundError:
    print(f"FEHLER: Projektpfad '{project_path}' nicht gefunden. Bitte Pfad prüfen.")
    # Fallback auf aktuelles Verzeichnis
    project_path = os.getcwd()
    print(f"Fallback Arbeitsverzeichnis: {project_path}")
except Exception as e:
    print(f"Fehler beim Wechseln des Verzeichnisses: {e}")
    project_path = os.getcwd() # Fallback
    print(f"Fallback Arbeitsverzeichnis: {project_path}")


# --- Installationen ---
print("\nInstalliere/Überprüfe Abhängigkeiten...")
# ipywidgets für HTML-Anzeige etc.
!pip install ipywidgets -q
# Gymnasium für RL Umgebung
!pip install gymnasium -q
# Stable-Baselines3 für RL Algorithmen
!pip install stable-baselines3[extra] -q # [extra] für TensorBoard etc.
print("Abhängigkeiten installiert/überprüft.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Aktuelles Arbeitsverzeichnis: /content/drive/MyDrive/Colab Notebooks

Installiere/Überprüfe Abhängigkeiten...
Abhängigkeiten installiert/überprüft.


# 2: Basis-Imports & Modul-Imports

In [ ]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 2: Basis-Imports & Modul-Imports (V5 - Mit Unmount/Remount & Modul-Löschen)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

print("\nImportiere Basis-Bibliotheken und eigene Module...")
import os
import sys
import random
import time
import math
import datetime
import numpy as np
from collections import defaultdict
from typing import Optional, Dict, List, Union, Tuple, Any
import traceback
import importlib # Bleibt wichtig für den Import

# Wichtig für HTML UI und Fortschrittsanzeige
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

# --- 1. Google Drive Unmount/Mount ---
try:
    from google.colab import drive
    # --- VERSUCHE ZUERST ZU UNMOUNTEN ---
    print("\nVersuche Google Drive zu unmounten (falls bereits gemountet)...")
    try:
        drive.flush_and_unmount()
        print("Drive erfolgreich unmounted.")
    except Exception as e:
        # Fehler hier ist oft okay, wenn Drive nicht gemountet war
        print(f"  Info/Fehler beim Unmounten: {e} (Kann ignoriert werden, wenn Drive nicht gemountet war)")
    # --- ENDE UNMOUNT ---

    # --- JETZT NEU MOUNTEN ---
    print("\nVersuche Google Drive neu zu mounten...")
    drive.mount('/content/drive', force_remount=True) # force_remount ist hier wichtig
    print("Google Drive gemountet.")
    # --- ENDE MOUNT ---

    # !!! WICHTIG: Pfad zu deinem Projektordner in Google Drive anpassen !!!
    project_path = '/content/drive/MyDrive/Colab Notebooks/' # Beispielpfad, ANPASSEN!

    # Überprüfe, ob der Pfad existiert
    if not os.path.isdir(project_path):
        print(f"FEHLER: Projektpfad '{project_path}' nicht gefunden. Bitte Pfad prüfen.")
        print(f"Verwende aktuelles Verzeichnis als Fallback: {os.getcwd()}")
        project_path = os.getcwd() # Fallback, wenn der Pfad nicht stimmt
    else:
        print(f"Projektpfad gesetzt auf: {project_path}")
        # Optional: Ins Verzeichnis wechseln (kann manchmal helfen)
        try:
            os.chdir(project_path)
            print(f"Arbeitsverzeichnis geändert zu: {os.getcwd()}")
        except Exception as e:
            print(f"Fehler beim Wechseln des Verzeichnisses: {e}")


except ImportError:
    print("Google Colab Umgebung nicht erkannt. Arbeite im aktuellen Verzeichnis.")
    project_path = os.getcwd()
    print(f"Aktuelles Arbeitsverzeichnis: {project_path}")
except Exception as e:
    print(f"Genereller Fehler beim Mounten/Unmounten von Google Drive: {e}")
    project_path = os.getcwd() # Fallback
    print(f"Fallback Arbeitsverzeichnis: {project_path}")


# --- 2. Hilfsfunktion zum Prüfen des Modul-Zeitstempels ---
def check_module_timestamp(module_name: str, module_obj: Optional[Any]) -> None:
    """ Prüft, ob ein Modul geladen wurde und gibt dessen Änderungsdatum aus. """
    if module_obj is None:
        print(f"FEHLER: Modul '{module_name}' konnte nicht importiert werden.")
        return
    try:
        file_path = getattr(module_obj, '__file__', None)
        if file_path and os.path.exists(file_path):
            timestamp = os.path.getmtime(file_path)
            mod_time = datetime.datetime.fromtimestamp(timestamp)
            print(f"  -> Modul '{module_name}' Info: Letzte Änderung: {mod_time.strftime('%Y-%m-%d %H:%M:%S')} (Datei: {os.path.basename(file_path)})")
        elif file_path:
            print(f"  -> Modul '{module_name}' geladen, aber Datei nicht gefunden: {file_path}")
        else:
            print(f"  -> Modul '{module_name}' geladen, aber Pfad (__file__) nicht verfügbar.")
    except Exception as e:
        print(f"  -> Fehler beim Abrufen des Zeitstempels für Modul '{module_name}': {e}")

# --- 3. Eigene Module importieren und prüfen ---
# Füge Projektpfad zu sys.path hinzu, falls noch nicht geschehen
if project_path not in sys.path:
    sys.path.insert(0, project_path)
    print(f"Projektpfad '{project_path}' zu sys.path hinzugefügt.")

# Liste der zu importierenden Module
module_names = [
    'rpg_config',
    'rpg_definitions',
    'rpg_game_logic',
    'rpg_ui',
    'rpg_env',
    'rpg_training_utils'
]
imported_modules = {}

print("\nVersuche, eigene Module zu importieren (mit Cache-Löschung) und Zeitstempel zu prüfen:")
all_imports_successful = True

for name in module_names:
    print(f"\nVerarbeite Modul: '{name}'...")
    # Explizites Löschen aus sys.modules
    if name in sys.modules:
        print(f"  - Entferne altes Modul '{name}' aus sys.modules...")
        try:
            del sys.modules[name]
            print(f"  - Modul '{name}' erfolgreich entfernt.")
        except KeyError:
            print(f"  - Warnung: Modul '{name}' war bereits aus sys.modules entfernt?")
    else:
        print(f"  - Modul '{name}' war nicht in sys.modules.")

    module = None
    try:
        # Importiere das Modul frisch
        module = importlib.import_module(name)
        print(f"  - Modul '{name}' neu importiert.")
        imported_modules[name] = module # Speichere das Modulobjekt
        # Prüfe den Zeitstempel direkt nach dem Import
        check_module_timestamp(name, module)
    except ImportError as e:
        print(f"\nFEHLER: Konnte Modul '{name}' nicht importieren: {e}")
        print("  -> Stelle sicher, dass die .py Datei existiert und keine Syntaxfehler enthält.")
        print("  -> Überprüfe ggf. Abhängigkeiten innerhalb des Moduls.")
        imported_modules[name] = None # Markiere als fehlgeschlagen
        all_imports_successful = False
    except Exception as e:
        print(f"\nFEHLER: Unerwarteter Fehler beim Import/Prüfen von Modul '{name}': {e}")
        traceback.print_exc()
        imported_modules[name] = None
        all_imports_successful = False

if all_imports_successful:
     print("\nAlle eigenen Module erfolgreich importiert/neu geladen und geprüft.")
else:
     print("\nWARNUNG: Mindestens ein Modul konnte nicht korrekt importiert werden. Folgefehler wahrscheinlich.")
     # Optional: Hier abbrechen?
     # raise ImportError("Ein oder mehrere Module konnten nicht geladen werden.")

# Versuche, die Module direkt in den globalen Namespace zu bringen
try:
    rpg_config = imported_modules.get('rpg_config')
    rpg_definitions = imported_modules.get('rpg_definitions')
    rpg_game_logic = imported_modules.get('rpg_game_logic')
    rpg_ui = imported_modules.get('rpg_ui')
    rpg_env = imported_modules.get('rpg_env')
    rpg_training_utils = imported_modules.get('rpg_training_utils')
    print("Module globalen Variablen zugewiesen.")
except Exception as e:
    print(f"Fehler beim Zuweisen der Module zu globalen Variablen: {e}")


# --- 4. RL Bibliotheken Imports ---
try:
    import gymnasium as gym
    from stable_baselines3 import PPO
    from stable_baselines3.common.env_checker import check_env
    print("\nRL-Bibliotheken importiert.")
except ImportError as e:
    print(f"\nFEHLER: RL-Bibliotheken nicht gefunden: {e}")
    print("Stelle sicher, dass die Installation in Zelle 1 erfolgreich war.")
    raise e

print("\n--- Ende Zelle 2 ---")


Importiere Basis-Bibliotheken und eigene Module...

Versuche Google Drive zu unmounten (falls bereits gemountet)...
Drive erfolgreich unmounted.

Versuche Google Drive neu zu mounten...
Mounted at /content/drive
Google Drive gemountet.
Projektpfad gesetzt auf: /content/drive/MyDrive/Colab Notebooks/
Arbeitsverzeichnis geändert zu: /content/drive/MyDrive/Colab Notebooks

Versuche, eigene Module zu importieren (mit Cache-Löschung) und Zeitstempel zu prüfen:

Verarbeite Modul: 'rpg_config'...
  - Entferne altes Modul 'rpg_config' aus sys.modules...
  - Modul 'rpg_config' erfolgreich entfernt.
Konfigurationsdatei rpg_config.py geschrieben/überschrieben (V4 - Ordner umbenannt).
  - LOAD_EXISTING_MODELS: False
  - Log Verzeichnis: /content/drive/MyDrive/Colab Notebooks/logs/ppo_klassen_lvl_v2/
  - Modell Verzeichnis: /content/drive/MyDrive/Colab Notebooks/models/klassen_agenten_lvl_v2/
  - Bericht Verzeichnis: /content/drive/MyDrive/Colab Notebooks/Auswertungsberichte/
  - Modul 'rpg_config

In [ ]:
print("Keys in geladenem rpg_definitions.CHAR_PARAMS:", list(rpg_definitions.CHAR_PARAMS.keys()))

Keys in geladenem rpg_definitions.CHAR_PARAMS: ['Krieger', 'Magier', 'Schurke', 'Kleriker', 'Goblin', 'Skelett', 'Ork']


# 3: Initialisierung der Spieldaten (Phase 3 - Objekte erstellen)


In [ ]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 3: Initialisierung der Spieldaten (Angepasst, V2)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import traceback # Sicherstellen, dass traceback für except Block verfügbar ist

print("\nInitialisiere Spieldaten (Skills)...")

# Stelle sicher, dass Module geladen sind
if 'rpg_definitions' not in locals() or 'rpg_game_logic' not in locals():
     raise ImportError("Benötigte Module (rpg_definitions, rpg_game_logic) nicht geladen. Bitte Zelle 2 ausführen.")

INITIALIZED_SKILLS: Dict[str, rpg_game_logic.Skill] = {}

def initialize_game_data(skill_params_dict: dict) -> Dict[str, rpg_game_logic.Skill]:
    """
    Erstellt Skill-Objekte aus den Parameter-Dictionaries.
    """
    initialized_skills = {}
    print(f"Initialisiere {len(skill_params_dict)} Skills...")
    for skill_name, params in skill_params_dict.items():
        try:
            # Erstelle Skill-Objekt mit den Parametern aus dem Dictionary
            skill_obj = rpg_game_logic.Skill(**params)
            initialized_skills[skill_name] = skill_obj
            # print(f"  - Skill '{skill_name}' initialisiert.")
        except TypeError as e:
            print(f"FEHLER bei Initialisierung von Skill '{skill_name}': {e}")
            print(f"  -> Übergebene Parameter: {params}")
        except Exception as e:
            print(f"Allgemeiner FEHLER bei Initialisierung von Skill '{skill_name}': {e}")
            traceback.print_exc()
    print(f"{len(initialized_skills)} Skills erfolgreich initialisiert.")
    return initialized_skills

# Führe die Initialisierung durch
try:
    INITIALIZED_SKILLS = initialize_game_data(rpg_definitions.SKILL_PARAMS)

    # Überprüfe kurz, ob wichtige Definitionen geladen wurden (angepasst)
    print("Prüfe wichtige Definitionen...")
    definitions_ok = True
    if not getattr(rpg_definitions, 'CHAR_PARAMS', None): # Sicherer Zugriff
        print("WARNUNG: CHAR_PARAMS aus rpg_definitions ist leer oder nicht vorhanden!")
        definitions_ok = False
    # Prüfe auf ALL_SKILLS_MAP statt SKILLS_BY_CLASS
    if not getattr(rpg_definitions, 'ALL_SKILLS_MAP', None): # Sicherer Zugriff
        print("WARNUNG: ALL_SKILLS_MAP aus rpg_definitions ist leer oder nicht vorhanden!")
        definitions_ok = False
    if not getattr(rpg_definitions, 'ALL_BUFFS_DEBUFFS_NAMES', None): # Sicherer Zugriff
         print("WARNUNG: ALL_BUFFS_DEBUFFS_NAMES aus rpg_definitions ist leer oder nicht vorhanden!")
         definitions_ok = False

    if definitions_ok:
        print("Wichtige Definitionen scheinen vorhanden zu sein.")
    else:
        print("-> Mindestens eine wichtige Definition fehlt in rpg_definitions.py!")

    print("\nSpieldaten initialisiert.")

except AttributeError as e:
    # Dieser Fehler sollte jetzt nicht mehr wegen SKILLS_BY_CLASS auftreten
    print(f"\nFEHLER (AttributeError) bei Initialisierung: {e}")
    print("Stelle sicher, dass rpg_definitions.py und rpg_game_logic.py korrekt sind.")
    traceback.print_exc()
except Exception as e:
     print(f"\nFEHLER bei der Initialisierung der Spieldaten: {e}")
     traceback.print_exc()


Initialisiere Spieldaten (Skills)...
Initialisiere 17 Skills...
17 Skills erfolgreich initialisiert.
Prüfe wichtige Definitionen...
Wichtige Definitionen scheinen vorhanden zu sein.

Spieldaten initialisiert.


# 4: RL Umgebung Instanziieren & Prüfen

In [ ]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 4: RL Umgebung Instanziieren & Prüfen (Angepasst für RPGEnv V6/V7/V8)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

print("\nErstelle und prüfe die RL-Umgebung...")

# Stelle sicher, dass Module geladen sind (aus Zelle 2)
# Füge hier einen expliziten traceback Import hinzu, falls er gebraucht wird
import traceback
if 'rpg_env' not in locals() or 'rpg_definitions' not in locals():
    raise ImportError("Benötigte Module (rpg_env, rpg_definitions) nicht geladen. Bitte Zelle 2 ausführen.")

rpg_environment_instance = None # Vorinitialisieren

try:
    # Erstelle eine Instanz der Umgebung
    # NUR noch char_param_definitions und optionale Parameter übergeben!
    print("--> Zelle 4: Aufruf RPGEnv(...) beginnt...")
    rpg_environment_instance = rpg_env.RPGEnv(
        char_param_definitions=rpg_definitions.CHAR_PARAMS,
        # skills_by_class=... <-- Nicht mehr benötigt, entfernt
        # initialized_skills=... <-- Nicht mehr benötigt, entfernt
        all_buffs_debuffs_names=rpg_definitions.ALL_BUFFS_DEBUFFS_NAMES,
        max_buff_duration=rpg_definitions.MAX_BUFF_DURATION,
        max_stacks=rpg_definitions.MAX_STACKS,
        render_mode='ansi' # oder None oder 'human' (falls UI anders gehandhabt wird)
    )
    print("--> Zelle 4: Aufruf RPGEnv(...) beendet.")
    print("RPGEnv Instanz erfolgreich erstellt.")

    # --- Umgebungs-Check (Optional, aber empfohlen) ---
    # Prüft, ob die Umgebung der Gymnasium API entspricht.
    print("Führe Gymnasium Env Check durch...")
    try:
        # Stelle sicher, dass check_env importiert wurde (normalerweise in Zelle 2)
        if 'check_env' not in locals():
            from stable_baselines3.common.env_checker import check_env
            print("Hinweis: stable_baselines3.common.env_checker.check_env nachimportiert.")

        check_env(rpg_environment_instance, warn=True)
        print(">>> Gymnasium Env Check erfolgreich abgeschlossen.")
    except NameError:
        print("FEHLER: check_env nicht gefunden. Konnte Env Check nicht durchführen.")
        print("        Stelle sicher, dass Zelle 2 gelaufen ist und stable_baselines3 importiert wurde.")
    except Exception as e:
        print(f"\nFEHLER beim Gymnasium Env Check: {e}")
        print("------------------------------------------------------------")
        print("Mögliche Ursachen:")
        print("- Inkonsistenzen zwischen Action/Observation Space Definition und step()/reset() Rückgaben.")
        print("- Falsche Datentypen (z.B. nicht numpy array, falscher dtype).")
        print("- Fehler in der reset() oder step() Logik der Umgebung.")
        print("Details:")
        traceback.print_exc() # Behalte Traceback hier bei
        print("------------------------------------------------------------")
        # Umgebung trotzdem behalten, um ggf. manuell zu debuggen

except AttributeError as e:
    # Dieser Fehler sollte jetzt nicht mehr auftreten, da SKILLS_BY_CLASS nicht mehr verwendet wird
    print(f"\nFEHLER: Benötigte Definitionen für RPGEnv nicht gefunden (z.B. aus rpg_definitions): {e}")
    traceback.print_exc() # Zeige den Traceback
    # raise e # Ggf. Fehler weiterwerfen, um Stopp zu erzwingen
except Exception as e:
     print(f"\nFEHLER beim Erstellen oder Prüfen der RPGEnv Instanz: {e}")
     traceback.print_exc()
     # raise e # Ggf. Fehler weiterwerfen


Erstelle und prüfe die RL-Umgebung...
--> Zelle 4: Aufruf RPGEnv(...) beginnt...
RPGEnv Instanz erstellt (V22 - Added is_success): ActionSpace=19, ObsSpace=51
--> Zelle 4: Aufruf RPGEnv(...) beendet.
RPGEnv Instanz erfolgreich erstellt.
Führe Gymnasium Env Check durch...
Skelett Magier_L1_499 benutzt 'Frostpfeil' auf Krieger_Hero.
Krieger_Hero erleidet 5 Schaden.
Goblin Krieger_L1_756 benutzt 'Keulenschlag' auf Magier_Hero.
Magier_Hero erleidet 17 Schaden.
Magier_Hero benutzt 'Schwächen' auf Goblin Krieger_L1_756.
Effekt 'Schwächen' wird auf Goblin Krieger_L1_756 angewendet (Dauer: 3, Mods: {'mod_angriff': -7, 'mod_verteidigung': -5}).
Goblin Krieger_L1_756 benutzt 'Keulenschlag' auf Magier_Hero.
Magier_Hero erleidet 10 Schaden.
Goblin Krieger_L1_756 benutzt 'Keulenschlag' auf Magier_Hero.
Magier_Hero erleidet 10 Schaden.
Effekt 'Schwächen' auf Goblin Krieger_L1_756 ist abgelaufen.
Goblin Krieger_L1_756 benutzt 'Keulenschlag' auf Magier_Hero.
Magier_Hero erleidet 17 Schaden.
Goblin Kr

In [ ]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 4.1: Funktion zum Anzeigen der Umgebungsinformationen
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

# Importiere die Klasse, um den Typ-Hint zu ermöglichen (optional aber gut)
try:
    from rpg_env import RPGEnv
except ImportError:
    class RPGEnv: pass # Dummy für den Fall, dass etwas schiefgeht

def print_env_details(env: RPGEnv):
    """ Gibt detaillierte Informationen über die Action/Observation Spaces der Env aus. """
    if not isinstance(env, RPGEnv):
        print("FEHLER: Übergebenes Objekt ist keine RPGEnv Instanz.")
        return

    print("\nRPG Umgebung Details:")
    print("-" * 50)

    # Action Space Details
    try:
        action_size = env.action_space.n
        # Verwende die action_to_name_map für die Ausgabe
        action_map = getattr(env, 'action_to_name_map', {})
        skills = [name for idx, name in action_map.items() if name not in ["Passen", "Basis-Angriff"]]
        num_skills = len(skills)
        pass_idx = getattr(env, '_pass_action_idx', 'N/A')
        basic_idx = getattr(env, '_basic_attack_idx', 'N/A')

        print(f"=> Aktionsmöglichkeiten (Action Space - Größe: {action_size}):")
        if skills:
             print(f"   - Skills (Indizes 0-{num_skills-1}): {skills}")
        else:
             print("   - Keine Skills gefunden.")
        print(f"   - Passen (Index {pass_idx})")
        print(f"   - Basis-Angriff (Index {basic_idx})")

    except AttributeError as e:
        print(f"Fehler beim Abrufen der Action Space Details: {e}")

    # Observation Space Details
    print("\n=> Beobachtungen (Observation Space - Größe: {}):".format(env.observation_space.shape[0] if hasattr(env.observation_space, 'shape') else 'Unbekannt'))
    try:
        num_res = 8
        num_cd = getattr(env, '_num_skills', len(getattr(env, 'all_skill_objects_list', [])))
        num_buff_features = 0
        buff_list = getattr(env, 'buff_list_for_obs', [])
        if buff_list:
             num_buff_features = len(buff_list) * 2 * 2

        print(f"   - {num_res} Werte: Ressourcen % [0..1] (HP, Mana, Stam, Energy für Held & Gegner)")
        print(f"   - {num_cd} Werte: Skill Cooldowns % [0..1] (für Held, Reihenfolge wie Skill-Liste)")
        print(f"   - {num_buff_features} Werte: Buffs/Debuffs [0..1] (Dauer % & Stacks % für Held & Gegner)")
        if buff_list:
            print(f"     -> Für Effekte: {buff_list}")
        else:
             print("     -> Keine Buff-/Debuff-Effekte im Observation Space definiert.")

    except AttributeError as e:
        print(f"Fehler beim Abrufen der Observation Space Details: {e}")

    print("-" * 50)

# Diese Zelle definiert nur die Funktion, sie wird hier noch nicht aufgerufen.
print("Funktion 'print_env_details' definiert.")

Funktion 'print_env_details' definiert.


# 5: Training der klassenspezifischen Agenten


In [ ]:
# -*- coding: utf-8 -*-
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 5: Training der klassenspezifischen Agenten mit IPyWidget Callback
#          (Angepasst: Debug-Ausgaben in separates Output-Widget umgeleitet)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

# --- Erforderliche Imports ---
import os
import numpy as np
import pandas as pd
import datetime
import traceback
from stable_baselines3.common.results_plotter import load_results
from stable_baselines3.common.monitor import Monitor
from stable_baselines3 import PPO
import importlib
import time
import sys # Für sys.modules Check
# *** HIER DIE ÄNDERUNG: Explizite Widget-Imports ***
import ipywidgets as widgets
from IPython.display import display

# --- Stelle sicher, dass Module aus vorherigen Zellen geladen sind ---
try:
    # Überprüfe, ob die notwendigen Module und Variablen existieren
    if 'rpg_config' not in globals(): raise NameError("Modul 'rpg_config' nicht gefunden.")
    if 'rpg_definitions' not in globals(): raise NameError("Modul 'rpg_definitions' nicht gefunden.")
    if 'rpg_env' not in globals(): raise NameError("Modul 'rpg_env' nicht gefunden.")
    if 'rpg_training_utils' not in globals(): raise NameError("Modul 'rpg_training_utils' nicht gefunden.")

    # Überprüfe, ob die benötigten Klassen/Variablen/Funktionen existieren
    if not hasattr(rpg_env, 'RPGEnv'): raise AttributeError("Klasse 'RPGEnv' nicht in Modul 'rpg_env' gefunden.")
    if not hasattr(rpg_definitions, 'CHAR_PARAMS'): raise AttributeError("Variable 'CHAR_PARAMS' nicht in Modul 'rpg_definitions' gefunden.")
    if not hasattr(rpg_config, 'LOG_DIR_BASE'): raise AttributeError("Variable 'LOG_DIR_BASE' nicht in Modul 'rpg_config' gefunden.")
    if not hasattr(rpg_config, 'SUMMARY_SAVE_DIR'): raise AttributeError("Variable 'SUMMARY_SAVE_DIR' nicht in Modul 'rpg_config' gefunden.")
    if not hasattr(rpg_config, 'MODEL_DIR_BASE'): raise AttributeError("Variable 'MODEL_DIR_BASE' nicht in Modul 'rpg_config' gefunden.")
    if not hasattr(rpg_config, 'TOTAL_TIMESTEPS_PER_CLASS'): raise AttributeError("Variable 'TOTAL_TIMESTEPS_PER_CLASS' nicht in Modul 'rpg_config' gefunden.")
    if not hasattr(rpg_config, 'CLASSES_TO_TRAIN'): raise AttributeError("Variable 'CLASSES_TO_TRAIN' nicht in Modul 'rpg_config' gefunden.")
    if not hasattr(rpg_config, 'LOAD_EXISTING_MODELS'): raise AttributeError("Variable 'LOAD_EXISTING_MODELS' nicht in Modul 'rpg_config' gefunden.")
    if not hasattr(rpg_config, 'DEFAULT_PPO_PARAMS'): raise AttributeError("Variable 'DEFAULT_PPO_PARAMS' nicht in Modul 'rpg_config' gefunden.")
    if not hasattr(rpg_training_utils, 'generate_layperson_log'):
        raise NameError("Funktion 'generate_layperson_log' NICHT in 'rpg_training_utils' gefunden.")
    # Überprüfe, ob die Callback-Klasse existiert
    if not hasattr(rpg_training_utils, 'IPyWidgetProgressCallback'):
        raise NameError("Klasse 'IPyWidgetProgressCallback' NICHT in 'rpg_training_utils' gefunden.")


except (NameError, ValueError, AttributeError) as e:
    print(f"FEHLER: Benötigte Module/Variablen/Attribute aus vorherigen Zellen fehlen: {e}")
    print("Bitte stelle sicher, dass die Zellen 1-4 erfolgreich ausgeführt wurden und die Module aktuell sind.")
    raise RuntimeError("Abhängigkeiten für Zelle 5 nicht erfüllt.") from e


# --- Hilfsfunktion für den Trainings-Loop (ANGEPASST für Debug-Widget) ---
# *** HIER DIE ÄNDERUNG: Übergabe des Debug-Widgets ***
def run_training_and_log_report(class_name, total_timesteps, model_dir, log_dir_base, summary_dir, debug_output_widget, load_existing=False):
    """
    Führt das Training für eine Klasse durch und generiert den verständlichen Textbericht.
    Nutzt den IPyWidgetProgressCallback und leitet Debug-Prints in debug_output_widget um.
    """
    print(f"\n{'='*20} Starte Training & Logging für: {class_name} {'='*20}")

    monitor_log_dir = os.path.join(log_dir_base, f"{class_name.lower()}_monitor_logs")
    os.makedirs(monitor_log_dir, exist_ok=True)
    print(f"Monitor Logs werden gespeichert in: {monitor_log_dir}")

    # Zielordner für Trainingsberichte definieren
    training_report_base_dir = os.path.join(summary_dir, "Training Berichte")
    today_str = datetime.date.today().strftime('%Y-%m-%d')
    daily_training_report_dir = os.path.join(training_report_base_dir, today_str)
    os.makedirs(daily_training_report_dir, exist_ok=True)
    print(f"  Tagesordner für Trainingsberichte: {daily_training_report_dir}")

    env_for_training = None
    model = None
    try:
        # 1. Eigene Umgebung erstellen und mit Monitor wrappen
        print("Erstelle NEUE Env-Instanz nur für diesen Trainingslauf...")
        train_env_instance = rpg_env.RPGEnv(
            char_param_definitions=rpg_definitions.CHAR_PARAMS,
            all_buffs_debuffs_names=rpg_definitions.ALL_BUFFS_DEBUFFS_NAMES,
            max_buff_duration=rpg_definitions.MAX_BUFF_DURATION,
            max_stacks=rpg_definitions.MAX_STACKS
        )
        train_env_instance.set_fixed_class(class_name)
        env_for_training = Monitor(train_env_instance, monitor_log_dir, allow_early_resets=True, info_keywords=('is_success',))
        print(f"Neue, gemonitorte Env für Training '{class_name}' erstellt.")

        # 2. Modell erstellen oder laden
        model_path = os.path.join(model_dir, f"ppo_{class_name.lower()}_agent.zip")
        current_steps = 0
        if load_existing and os.path.exists(model_path):
            try:
                print(f"Lade existierendes Modell: {model_path}")
                model = PPO.load(model_path, env=env_for_training)
                current_steps = getattr(model, 'num_timesteps', 0)
                print(f"Modell geladen. Bisherige Schritte: {current_steps}")
            except Exception as e:
                print(f"FEHLER beim Laden von {model_path}: {e}. Erstelle neues Modell.")
                model = None; current_steps = 0
        if model is None:
            print("Erstelle neues PPO-Modell.")
            params = rpg_config.DEFAULT_PPO_PARAMS.copy()
            params['tensorboard_log'] = None
            if 'policy' not in params: params['policy'] = 'MlpPolicy'
            model = PPO(env=env_for_training, **params)
            current_steps = 0

        # 3. Training starten
        remaining_timesteps = total_timesteps - current_steps
        if remaining_timesteps > 0:
            print(f"Starte Training von {current_steps} bis {total_timesteps}...")

            callback_instance = None
            try:
                callback_instance = rpg_training_utils.IPyWidgetProgressCallback(total_timesteps=remaining_timesteps)
            except Exception as e:
                 print(f"WARNUNG: Konnte IPyWidgetProgressCallback nicht erstellen: {e}")

            # *** HIER DIE ÄNDERUNG: model.learn im Kontext des Debug-Widgets ausführen ***
            with debug_output_widget:
                # Alle print() ausgaben innerhalb dieses Blocks (also aus env.step)
                # werden in das debug_output_widget umgeleitet.
                print(f"--- Debug Output für {class_name} ---") # Header für das Widget
                model.learn(
                    total_timesteps=remaining_timesteps,
                    callback=callback_instance,
                    reset_num_timesteps=(not load_existing),
                    progress_bar=False
                )
            # ***************************************************************************

            print("Training abgeschlossen.") # Diese Ausgabe erscheint wieder normal

            # 4. Modell speichern
            latest_path = model_path
            try:
                print(f"Speichere Modell nach: {latest_path}")
                model.save(latest_path)
            except Exception as e:
                 print(f"FEHLER beim Speichern des Modells für {class_name}: {e}")
        else:
             print(f"Ziel-Timesteps ({total_timesteps}) bereits erreicht ({current_steps}). Überspringe Training.")

        # --- 5. Verständlichen Trainings-Bericht generieren ---
        print(f"\nErstelle verständlichen Textbericht für Training von {class_name}...")
        output_file = os.path.join(
            daily_training_report_dir,
            f"trainingsbericht_{class_name.lower()}_verstaendlich.txt"
        )
        config_params_for_log = rpg_config.DEFAULT_PPO_PARAMS

        rpg_training_utils.generate_layperson_log(
            log_dir=monitor_log_dir,
            output_filepath=output_file,
            class_name=class_name,
            total_timesteps=total_timesteps,
            config_params=config_params_for_log
        )

    except Exception as e:
        print(f"\nFEHLER im Trainings-/Logging-Prozess für Klasse {class_name}: {e}")
        # *** HIER DIE ÄNDERUNG: Fehler auch ins Debug-Widget schreiben ***
        with debug_output_widget:
             print(f"\nFEHLER im Trainings-/Logging-Prozess für Klasse {class_name}: {e}")
             traceback.print_exc()
        # ****************************************************************
    finally:
        # 6. Umgebung schließen
        if env_for_training is not None:
            env_for_training.close()
            print(f"Trainingsumgebung für {class_name} geschlossen.")

    print(f"{'='*20} Training & Logging für {class_name} beendet {'='*20}\n")


# --- Haupt-Trainingsschleife ---

# *** HIER DIE ÄNDERUNG: Output Widget erstellen und anzeigen ***
print("Erstelle separates Widget für Debug-Ausgaben...")
debug_output_widget = widgets.Output(layout={
    'border': '1px solid grey',
    'height': '350px', # Höhe anpassen nach Bedarf
    'overflow_y': 'scroll' # Scrollbar hinzufügen
})
display(debug_output_widget) # Widget anzeigen
# ************************************************************


print(f"\n=== Starte Trainingsprozess für Klassen: {rpg_config.CLASSES_TO_TRAIN} ===")
print(f"Gesamt-Timesteps pro Klasse: {rpg_config.TOTAL_TIMESTEPS_PER_CLASS}")
print(f"Modelle laden/speichern in: {rpg_config.MODEL_DIR_BASE}")
print(f"Basis-Log Verzeichnis (Monitor): {rpg_config.LOG_DIR_BASE}")
print(f"Basis-Verzeichnis für Berichte: {rpg_config.SUMMARY_SAVE_DIR}")
print(f"Existierende Modelle laden: {rpg_config.LOAD_EXISTING_MODELS}")
print("-" * 60)

start_time_total = time.time()

try:
    # Aufruf für jede Klasse
    for class_name in rpg_config.CLASSES_TO_TRAIN:
        if class_name not in rpg_definitions.CHAR_PARAMS:
             print(f"WARNUNG: Klasse '{class_name}' nicht in CHAR_PARAMS gefunden. Überspringe.")
             continue

        # *** HIER DIE ÄNDERUNG: Debug-Widget an die Funktion übergeben ***
        run_training_and_log_report(
            class_name=class_name,
            total_timesteps=rpg_config.TOTAL_TIMESTEPS_PER_CLASS,
            model_dir=rpg_config.MODEL_DIR_BASE,
            log_dir_base=rpg_config.LOG_DIR_BASE,
            summary_dir=rpg_config.SUMMARY_SAVE_DIR,
            debug_output_widget=debug_output_widget, # Widget übergeben
            load_existing=rpg_config.LOAD_EXISTING_MODELS
        )
        # ***************************************************************
        print("-" * 60) # Trennlinie zwischen Klassen

except NameError as e:
     print(f"\nFATALER FEHLER: Benötigte Variable/Modul nicht gefunden: {e}")
     print("Stelle sicher, dass Zellen 1-4 ausgeführt wurden und alle Module/Variablen korrekt definiert sind.")
     traceback.print_exc()
except Exception as e:
     print(f"\nFATALER Allgemeiner Fehler im Trainings-Loop: {e}")
     traceback.print_exc()
finally:
    end_time_total = time.time()
    duration_total = end_time_total - start_time_total
    print(f"\n=== Gesamter Trainingsprozess beendet ===")
    print(f"Gesamtdauer: {datetime.timedelta(seconds=duration_total)}")

Erstelle separates Widget für Debug-Ausgaben...


Output(layout=Layout(border='1px solid grey', height='350px', overflow_y='scroll'))


=== Starte Trainingsprozess für Klassen: ['Krieger', 'Magier', 'Schurke', 'Kleriker'] ===
Gesamt-Timesteps pro Klasse: 50000
Modelle laden/speichern in: /content/drive/MyDrive/Colab Notebooks/models/klassen_agenten_lvl_v2/
Basis-Log Verzeichnis (Monitor): /content/drive/MyDrive/Colab Notebooks/logs/ppo_klassen_lvl_v2/
Basis-Verzeichnis für Berichte: /content/drive/MyDrive/Colab Notebooks/Auswertungsberichte/
Existierende Modelle laden: False
------------------------------------------------------------

==================== Starte Training & Logging für: Krieger ====================
Monitor Logs werden gespeichert in: /content/drive/MyDrive/Colab Notebooks/logs/ppo_klassen_lvl_v2/krieger_monitor_logs
  Tagesordner für Trainingsberichte: /content/drive/MyDrive/Colab Notebooks/Auswertungsberichte/Training Berichte/2025-04-17
Erstelle NEUE Env-Instanz nur für diesen Trainingslauf...
RPGEnv Instanz erstellt (V22 - Added is_success): ActionSpace=19, ObsSpace=51
Neue, gemonitorte Env für Tra

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
Krieger_Hero erleidet 38 Schaden.
Krieger_Hero benutzt 'Mächtiger Schlag' auf Ork_L1_283.
Ork_L1_283 erleidet 36 Schaden.
Ork_L1_283 wurde besiegt!
Krieger_Hero erhält 25 XP. (Aktuell: 25/100)
Krieger_Hero benutzt 'Mächtiger Schlag' auf Goblin_L1_673.
Goblin_L1_673 erleidet 46 Schaden.
Goblin_L1_673 benutzt 'Keulenschlag' auf Krieger_Hero.
Krieger_Hero erleidet 2 Schaden.
Krieger_Hero benutzt 'Mächtiger Schlag' auf Goblin_L1_673.
Goblin_L1_673 erleidet 46 Schaden.
Goblin_L1_673 wurde besiegt!
Krieger_Hero erhält 25 XP. (Aktuell: 25/100)
Krieger_Hero benutzt 'Mächtiger Schlag' auf Goblin Krieger_L1_639.
Goblin Krieger_L1_639 erleidet 43 Schaden.
Goblin Krieger_L1_639 benutzt 'Keulenschlag' auf Krieger_Hero.
Krieger_Hero erleidet 6 Schaden.
Krieger_Hero benutzt 'Mächtiger Schlag' auf Goblin Krieger_L1_639.
Goblin Krieger_L1_639 erleidet 43 Schaden.
Goblin Krieger_L1_639 benutzt 'Keulenschlag' auf Krieger_Hero.
Krieger_Her

/usr/local/lib/python3.11/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):


Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
Goblin_L1_418 benutzt 'Keulenschlag' auf Magier_Hero.
Magier_Hero erleidet 6 Schaden.
Magier_Hero benutzt 'Feuerball' auf Goblin_L1_418.
Goblin_L1_418 erleidet 56 Schaden.
Goblin_L1_418 wurde besiegt!
Magier_Hero erhält 25 XP. (Aktuell: 25/100)
Magier_Hero benutzt 'Feuerball' auf Goblin_L1_390.
Goblin_L1_390 erleidet 51 Schaden.
Goblin_L1_390 benutzt 'Keulenschlag' auf Magier_Hero.
Magier_Hero erleidet 13 Schaden.
Goblin_L1_390 benutzt 'Keulenschlag' auf Magier_Hero.
Magier_Hero erleidet 13 Schaden.
Magier_Hero benutzt 'Feuerball' auf Goblin_L1_390.
Goblin_L1_390 erleidet 51 Schaden.
Goblin_L1_390 wurde besiegt!
Magier_Hero erhält 25 XP. (Aktuell: 25/100)
Skelett Magier_L1_638 benutzt 'Frostpfeil' auf Magier_Hero.
Magier_Hero erleidet 16 Schaden.
Magier_Hero benutzt 'Feuerball' auf Skelett Magier_L1_638.
Skelett Magier_L1_638 erleidet 55 Schaden.
Skelett Magier_L1_638 wurde besiegt!
Magier_Hero erhält 25 XP. (Aktuell: 2

/usr/local/lib/python3.11/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):


Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
Goblin Schamane_L1_654 benutzt 'Blitz' auf Schurke_Hero.
Schurke_Hero erleidet 6 Schaden.
Goblin Schamane_L1_654 erleidet 5 Schaden durch 'Vergiften'.
Schurke_Hero benutzt 'Ausweichen verbessern' auf Schurke_Hero.
Effekt 'Ausweichen verbessern' wird auf Schurke_Hero angewendet (Dauer: 4, Mods: {'mod_ausweichen': 40}).
Goblin Schamane_L1_654 erleidet 5 Schaden durch 'Vergiften'.
Schurke_Hero benutzt 'Schneller Stich' auf Goblin Schamane_L1_654.
Goblin Schamane_L1_654 erleidet 10 Schaden.
Effekt 'Schwächen' auf Schurke_Hero ist abgelaufen.
Goblin Schamane_L1_654 erleidet 5 Schaden durch 'Vergiften'.
Goblin Schamane_L1_654 benutzt 'Schwächen' auf Schurke_Hero.
Effekt 'Schwächen' wird auf Schurke_Hero angewendet (Dauer: 3, Mods: {'mod_angriff': -7, 'mod_verteidigung': -5}).
Goblin Schamane_L1_654 erleidet 5 Schaden durch 'Vergiften'.
Effekt 'Vergiften' auf Goblin Schamane_L1_654 ist abgelaufen.
Schurke_Hero benutzt 'Vergift

/usr/local/lib/python3.11/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):


Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
Effekt 'Segnen' wird auf Kleriker_Hero angewendet (Dauer: 4, Mods: {'mod_verteidigung': 5, 'mod_magiekraft': 3}).
Kleriker_Hero benutzt 'Leichte Heilung' auf Kleriker_Hero.
Kleriker_Hero wird um 63 HP geheilt.
Ork Berserker_L2_559 benutzt 'Wuchtschlag' auf Kleriker_Hero.
Kleriker_Hero erleidet 49 Schaden.
Kleriker_Hero benutzt 'Schwächen' auf Ork Berserker_L2_559.
Effekt 'Schwächen' wird auf Ork Berserker_L2_559 angewendet (Dauer: 3, Mods: {'mod_angriff': -7, 'mod_verteidigung': -5}).
Ork Berserker_L2_559 benutzt 'Wutanfall' auf Kleriker_Hero.
Effekt 'Wutanfall' wird auf Ork Berserker_L2_559 angewendet (Dauer: 3, Mods: {'mod_angriff': 8, 'mod_verteidigung': -5}).
Kleriker_Hero benutzt 'Heiliger Schlag' auf Ork Berserker_L2_559.
Ork Berserker_L2_559 erleidet 36 Schaden.
Ork Berserker_L2_559 wurde besiegt!
Effekt 'Segnen' auf Kleriker_Hero ist abgelaufen.
Kleriker_Hero erhält 63 XP. (Aktuell: 63/100)
Kleriker_Hero benutzt

# 6: TensorBoard starten


In [ ]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 6: TensorBoard starten (Korrigierter Pfad mit Anführungszeichen)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

print("\nStarte TensorBoard...")
# Laden der TensorBoard Notebook Erweiterung (sollte noch geladen sein, schadet aber nicht)
%load_ext tensorboard

# Stelle sicher, dass die Variable LOG_DIR_BASE aus rpg_config existiert
if 'rpg_config' in locals() and hasattr(rpg_config, 'LOG_DIR_BASE'):
    log_directory = rpg_config.LOG_DIR_BASE
    print(f"TensorBoard wird gestartet und überwacht das Verzeichnis: {log_directory}")
    # WICHTIG: Den Pfad in Anführungszeichen setzen!
    %tensorboard --logdir "{log_directory}"
else:
    print("FEHLER: LOG_DIR_BASE nicht in rpg_config gefunden. Kann TensorBoard nicht starten.")
    print("Bitte stelle sicher, dass Zelle 2 (Imports) erfolgreich gelaufen ist.")

# 7: Detaillierte Evaluierung der Agenten

In [ ]:
# -*- coding: utf-8 -*-
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 7: Detaillierte Evaluierung der Agenten
#          (Angepasst V3: Korrigierter Aufruf von evaluate_agent_and_summarize ohne combat_log_widget)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import traceback
import importlib # Bleibt für ggf. manuelle Reloads nützlich
import os
import sys # Für sys.modules Check

# --- Stelle sicher, dass Module aus vorherigen Zellen korrekt geladen sind ---
# Annahme: Zelle 2 hat die Module importiert und ggf. neu geladen!
try:
    print("Prüfe Verfügbarkeit der Module und benötigten Inhalte...")
    # Überprüfe, ob die Module im globalen Scope existieren
    if 'rpg_config' not in globals(): raise NameError("Modul 'rpg_config' nicht global gefunden.")
    if 'rpg_definitions' not in globals(): raise NameError("Modul 'rpg_definitions' nicht global gefunden.")
    if 'rpg_training_utils' not in globals(): raise NameError("Modul 'rpg_training_utils' nicht global gefunden.")
    # Stelle sicher, dass die NEUESTE Version von rpg_training_utils geladen ist
    if not hasattr(rpg_training_utils, '_save_report_html'): # Prüfe auf eine spätere Funktion als Indikator
         print("WARNUNG: rpg_training_utils scheint nicht aktuell zu sein (Funktion _save_report_html fehlt). Lade neu...")
         importlib.reload(rpg_training_utils)
         if not hasattr(rpg_training_utils, 'evaluate_agent_and_summarize'):
             raise NameError("Funktion 'evaluate_agent_and_summarize' auch nach Reload nicht gefunden.")


    # Überprüfe, ob die benötigten Variablen/Funktionen in den Modulen existieren
    if not hasattr(rpg_config, 'CLASSES_TO_TRAIN'): raise NameError("Variable 'CLASSES_TO_TRAIN' nicht in rpg_config gefunden.")
    if not hasattr(rpg_definitions, 'CHAR_PARAMS'): raise NameError("Variable 'CHAR_PARAMS' nicht in rpg_definitions gefunden.")
    if not hasattr(rpg_config, 'MODEL_DIR_BASE'): raise NameError("Variable 'MODEL_DIR_BASE' nicht in rpg_config gefunden.")
    if not hasattr(rpg_config, 'SUMMARY_SAVE_DIR'): raise NameError("Variable 'SUMMARY_SAVE_DIR' nicht in rpg_config gefunden.")
    if not hasattr(rpg_config, 'EVAL_DETERMINISTIC'): raise NameError("Variable 'EVAL_DETERMINISTIC' nicht in rpg_config gefunden.")
    if not hasattr(rpg_config, 'MAX_TURNS'): raise NameError("Variable 'MAX_TURNS' nicht in rpg_config gefunden.")
    if not hasattr(rpg_config, 'DEFAULT_PPO_PARAMS'): raise NameError("Variable 'DEFAULT_PPO_PARAMS' nicht in rpg_config gefunden.")
    if not hasattr(rpg_training_utils, 'evaluate_agent_and_summarize'):
         raise NameError("Funktion 'evaluate_agent_and_summarize' nicht in rpg_training_utils gefunden.")
    # Überprüfe die Konstante erneut
    if not hasattr(rpg_training_utils, 'N_FIGHTS_PER_OPPONENT'):
         raise NameError("Konstante 'N_FIGHTS_PER_OPPONENT' nicht in rpg_training_utils gefunden.")

    print("Alle benötigten Module und Inhalte scheinen verfügbar zu sein.")

except (NameError, ImportError, AttributeError) as e:
    print(f"FEHLER: Benötigte Module/Variablen/Funktionen nicht gefunden: {e}")
    print("Bitte stelle sicher, dass Zelle 2 erfolgreich und NACH allen '%%writefile'-Befehlen ausgeführt wurde.")
    raise RuntimeError("Abhängigkeiten für Zelle 7 nicht erfüllt.") from e
except Exception as e:
     print(f"FEHLER beim Prüfen der Module: {e}")
     traceback.print_exc()
     raise e


# --- Evaluierung starten ---
print(f"\n=== Starte detaillierte Evaluierung für Klassen: {rpg_config.CLASSES_TO_TRAIN} ===")
# Parameter-Ausgaben mit Werten aus rpg_config
print(f"Kämpfe pro Gegner: {rpg_training_utils.N_FIGHTS_PER_OPPONENT}") # Zugriff auf die Konstante
print(f"Modelle laden aus: {rpg_config.MODEL_DIR_BASE}")
print(f"Zusammenfassungen speichern in (Basisordner): {rpg_config.SUMMARY_SAVE_DIR}")
print(f"Aktionswahl: {'Festgelegt' if rpg_config.EVAL_DETERMINISTIC else 'Zufällig (mit Erkundung)'}")
print("-" * 60)


# --- Schleife für jede Klasse ---
for class_name in rpg_config.CLASSES_TO_TRAIN:
    if class_name not in rpg_definitions.CHAR_PARAMS:
         print(f"\nWARNUNG: Klasse '{class_name}' nicht in CHAR_PARAMS gefunden. Überspringe Evaluierung.")
         continue

    try:
        # --- KORRIGIERTER AUFRUF OHNE 'combat_log_widget' ---
        rpg_training_utils.evaluate_agent_and_summarize(
            class_name=class_name,
            model_dir_base=rpg_config.MODEL_DIR_BASE,
            summary_dir=rpg_config.SUMMARY_SAVE_DIR, # Basisordner übergeben
            deterministic=rpg_config.EVAL_DETERMINISTIC,
            max_turns_per_episode=rpg_config.MAX_TURNS,
            training_params=rpg_config.DEFAULT_PPO_PARAMS, # Optional übergeben
            output_format='txt_single' # Hier Ausgabeformat wählen (z.B. 'txt_single', 'csv', 'html')
        )
        # --- Ende korrigierter Funktionsaufruf ---

    except Exception as e:
        print(f"\nFEHLER bei der Evaluierung von Klasse {class_name}: {e}")
        traceback.print_exc()

    print("-" * 60) # Trennlinie

print(f"\n=== Detaillierte Evaluierung für alle Klassen abgeschlossen (oder versucht) ===")
print(f"Überprüfe die .txt/.csv/.html-Dateien im Ordner (bzw. dessen Tages-Unterordnern): {rpg_config.SUMMARY_SAVE_DIR}")

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
Kleriker_Hero benutzt 'Schwächen' auf Ork_L1_649.
Effekt 'Schwächen' wird auf Ork_L1_649 angewendet (Dauer: 3, Mods: {'mod_angriff': -7, 'mod_verteidigung': -5}).
Kleriker_Hero benutzt 'Heiliger Schlag' auf Ork_L1_649.
Ork_L1_649 erleidet 28 Schaden.
Ork_L1_649 benutzt 'Wuchtschlag' auf Kleriker_Hero.
Kleriker_Hero erleidet 39 Schaden.
Effekt 'Schwächen' auf Ork_L1_649 ist abgelaufen.
Kleriker_Hero benutzt 'Leichte Heilung' auf Kleriker_Hero.
Kleriker_Hero wird um 60 HP geheilt.
Kleriker_Hero benutzt 'Schwächen' auf Ork_L1_649.
Effekt 'Schwächen' wird auf Ork_L1_649 angewendet (Dauer: 3, Mods: {'mod_angriff': -7, 'mod_verteidigung': -5}).
Kleriker_Hero benutzt 'Heiliger Schlag' auf Ork_L1_649.
Ork_L1_649 erleidet 28 Schaden.
Ork_L1_649 benutzt 'Wuchtschlag' auf Kleriker_Hero.
Kleriker_Hero erleidet 39 Schaden.
Kleriker_Hero benutzt 'Regeneration' auf Kleriker_Hero.
Effekt 'Regeneration' wird auf Kleriker_Hero angewendet

In [ ]:
# -*- coding: utf-8 -*-
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 7: Detaillierte Evaluierung der Agenten
#          (Angepasst V2: Vereinfachter Import-Check, nutzt global geladene Module)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import traceback
import importlib # Bleibt für ggf. manuelle Reloads nützlich
import os
import sys # Für sys.modules Check

# --- Stelle sicher, dass Module aus vorherigen Zellen korrekt geladen sind ---
# Annahme: Zelle 2 hat die Module importiert und ggf. neu geladen!
try:
    print("Prüfe Verfügbarkeit der Module und benötigten Inhalte...")
    # Überprüfe, ob die Module im globalen Scope existieren
    if 'rpg_config' not in globals(): raise NameError("Modul 'rpg_config' nicht global gefunden.")
    if 'rpg_definitions' not in globals(): raise NameError("Modul 'rpg_definitions' nicht global gefunden.")
    if 'rpg_training_utils' not in globals(): raise NameError("Modul 'rpg_training_utils' nicht global gefunden.")

    # Überprüfe, ob die benötigten Variablen/Funktionen in den Modulen existieren
    if not hasattr(rpg_config, 'CLASSES_TO_TRAIN'): raise NameError("Variable 'CLASSES_TO_TRAIN' nicht in rpg_config gefunden.")
    if not hasattr(rpg_definitions, 'CHAR_PARAMS'): raise NameError("Variable 'CHAR_PARAMS' nicht in rpg_definitions gefunden.")
    if not hasattr(rpg_config, 'MODEL_DIR_BASE'): raise NameError("Variable 'MODEL_DIR_BASE' nicht in rpg_config gefunden.")
    if not hasattr(rpg_config, 'SUMMARY_SAVE_DIR'): raise NameError("Variable 'SUMMARY_SAVE_DIR' nicht in rpg_config gefunden.")
    if not hasattr(rpg_config, 'EVAL_DETERMINISTIC'): raise NameError("Variable 'EVAL_DETERMINISTIC' nicht in rpg_config gefunden.")
    if not hasattr(rpg_config, 'MAX_TURNS'): raise NameError("Variable 'MAX_TURNS' nicht in rpg_config gefunden.")
    if not hasattr(rpg_config, 'DEFAULT_PPO_PARAMS'): raise NameError("Variable 'DEFAULT_PPO_PARAMS' nicht in rpg_config gefunden.")
    if not hasattr(rpg_training_utils, 'evaluate_agent_and_summarize'):
         raise NameError("Funktion 'evaluate_agent_and_summarize' nicht in rpg_training_utils gefunden.")
    # Überprüfe die Konstante erneut
    if not hasattr(rpg_training_utils, 'N_FIGHTS_PER_OPPONENT'):
         raise NameError("Konstante 'N_FIGHTS_PER_OPPONENT' nicht in rpg_training_utils gefunden.")

    print("Alle benötigten Module und Inhalte scheinen verfügbar zu sein.")

except (NameError, ImportError, AttributeError) as e:
    print(f"FEHLER: Benötigte Module/Variablen/Funktionen nicht gefunden: {e}")
    print("Bitte stelle sicher, dass Zelle 2 erfolgreich und NACH allen '%%writefile'-Befehlen ausgeführt wurde.")
    raise RuntimeError("Abhängigkeiten für Zelle 7 nicht erfüllt.") from e
except Exception as e:
     print(f"FEHLER beim Prüfen der Module: {e}")
     traceback.print_exc()
     raise e


# --- Evaluierung starten ---
print(f"\n=== Starte detaillierte Evaluierung für Klassen: {rpg_config.CLASSES_TO_TRAIN} ===")
# Parameter-Ausgaben mit Werten aus rpg_config
print(f"Kämpfe pro Gegner: {rpg_training_utils.N_FIGHTS_PER_OPPONENT}") # Zugriff auf die Konstante
print(f"Modelle laden aus: {rpg_config.MODEL_DIR_BASE}")
print(f"Zusammenfassungen speichern in (Basisordner): {rpg_config.SUMMARY_SAVE_DIR}")
print(f"Aktionswahl: {'Festgelegt' if rpg_config.EVAL_DETERMINISTIC else 'Zufällig (mit Erkundung)'}")
print("-" * 60)


# --- Schleife für jede Klasse ---
for class_name in rpg_config.CLASSES_TO_TRAIN:
    if class_name not in rpg_definitions.CHAR_PARAMS:
         print(f"\nWARNUNG: Klasse '{class_name}' nicht in CHAR_PARAMS gefunden. Überspringe Evaluierung.")
         continue

    try:
        # --- Korrekter Aufruf für V19+ von evaluate_agent_and_summarize ---
        rpg_training_utils.evaluate_agent_and_summarize(
            class_name=class_name,
            model_dir_base=rpg_config.MODEL_DIR_BASE,
            summary_dir=rpg_config.SUMMARY_SAVE_DIR, # Basisordner übergeben
            deterministic=rpg_config.EVAL_DETERMINISTIC,
            max_turns_per_episode=rpg_config.MAX_TURNS,
            training_params=rpg_config.DEFAULT_PPO_PARAMS, # Optional übergeben
            output_format='txt_single' # Hier Ausgabeformat wählen
        )
        # --- Ende Funktionsaufruf ---

    except Exception as e:
        print(f"\nFEHLER bei der Evaluierung von Klasse {class_name}: {e}")
        traceback.print_exc()

    print("-" * 60) # Trennlinie

print(f"\n=== Detaillierte Evaluierung für alle Klassen abgeschlossen (oder versucht) ===")
print(f"Überprüfe die .txt/.csv/.html-Dateien im Ordner (bzw. dessen Tages-Unterordnern): {rpg_config.SUMMARY_SAVE_DIR}")

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
Kleriker_Hero benutzt 'Schwächen' auf Ork_L2_597.
Effekt 'Schwächen' wird auf Ork_L2_597 angewendet (Dauer: 3, Mods: {'mod_angriff': -7, 'mod_verteidigung': -5}).
Kleriker_Hero benutzt 'Segnen' auf Kleriker_Hero.
Effekt 'Segnen' wird auf Kleriker_Hero angewendet (Dauer: 4, Mods: {'mod_verteidigung': 5, 'mod_magiekraft': 3}).
Kleriker_Hero benutzt 'Heiliger Schlag' auf Ork_L2_597.
Ork_L2_597 erleidet 31 Schaden.
Ork_L2_597 benutzt 'Wuchtschlag' auf Kleriker_Hero.
Kleriker_Hero erleidet 34 Schaden.
Effekt 'Schwächen' auf Ork_L2_597 ist abgelaufen.
Kleriker_Hero benutzt 'Leichte Heilung' auf Kleriker_Hero.
Kleriker_Hero wird um 63 HP geheilt.
Effekt 'Segnen' auf Kleriker_Hero ist abgelaufen.
Kleriker_Hero benutzt 'Schwächen' auf Ork_L2_597.
Effekt 'Schwächen' wird auf Ork_L2_597 angewendet (Dauer: 3, Mods: {'mod_angriff': -7, 'mod_verteidigung': -5}).
Kleriker_Hero benutzt 'Heiliger Schlag' auf Ork_L2_597.
Ork_L2_597 erlei

# 8: Beispielhafte UI-Anzeige (Optional)

In [ ]:
print("\nOptional: Zeige ein Beispiel der HTML UI...")

if rpg_environment_instance:
    try:
        # Setze die Umgebung für eine Beispielklasse zurück
        example_class = random.choice(rpg_config.CLASSES_TO_TRAIN)
        rpg_environment_instance.set_fixed_class(example_class)
        obs, info = rpg_environment_instance.reset()

        # Erzeuge ein paar Beispiel-Logeinträge
        example_logs = [
            f"Kampf beginnt! {rpg_environment_instance.hero.name} vs {rpg_environment_instance.enemy.name}",
            f"{rpg_environment_instance.hero.name} setzt Feuerball ein.",
            f"{rpg_environment_instance.enemy.name} erleidet 35 Schaden.",
            f"{rpg_environment_instance.enemy.name} wendet Vergiften auf {rpg_environment_instance.hero.name} an.",
            f"{rpg_environment_instance.hero.name} erleidet 5 Schaden durch Gift.",
            f"{rpg_environment_instance.hero.name} wird um 30 geheilt.",
            f"{rpg_environment_instance.enemy.name} wurde besiegt!"
        ]

        # Generiere und zeige die UI
        html_output = rpg_ui.generate_html_ui(rpg_environment_instance, example_logs)
        print(f"Zeige UI für Beispielszene (Klasse: {example_class}):")
        display(html_output)

    except Exception as e:
        print(f"FEHLER beim Erzeugen der Beispiel-UI: {e}")
        traceback.print_exc()
else:
    print("Keine RPGEnv Instanz für UI-Beispiel verfügbar.")

# 9: Hilfsprogramme (Summary Generatoren)


In [ ]:
    # -*- coding: utf-8 -*-
    # +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    # ZELLE 9.1: Hilfsskript zum Generieren der Entscheidungs-Zusammenfassung (TXT)
    # +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

    import os
    import datetime

    # *** HIER DIE ÄNDERUNG: Standard-Dateiname auf .txt geändert ***
    def generate_decision_summary_txt(filename="ENTSCHEIDUNGEN_AKTUELL.txt"):
        """
        Generiert eine Text-Datei mit der detaillierten Zusammenfassung aller
        Projektentscheidungen.
        """
        print(f"Generiere detaillierte Entscheidungszusammenfassung nach '{filename}'...")

        # Hole den aktuellen Zeitstempel
        now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Der gesamte Text der Zusammenfassung, Markdown entfernt/vereinfacht
        summary_text = f"""Projektentscheidungen & Status: Python RPG (Stand: {now})
    =========================================================

    Dieses Dokument fasst die grundlegenden und aktuellen Design- und Implementierungsentscheidungen für das selbstlaufende Python-RPG-Projekt zusammen.

    1. Grundlegende Architektur & Umgebung
    ---------------------------------------
    - Ziel: Entwicklung eines selbstlaufenden RPGs in Python, primär zur Beobachtung von KI-Verhalten (Skript-basiert und Reinforcement Learning).
    - Entwicklungsumgebung: Google Colab mit einem zentralen Jupyter Notebook (RPG.ipynb).
    - Modularisierung: Code ist in separate .py-Dateien ausgelagert (rpg_config.py, rpg_definitions.py, rpg_game_logic.py, rpg_env.py, rpg_ui.py, rpg_training_utils.py) zur besseren Organisation und Wiederverwendbarkeit. Updates erfolgen oft via %%writefile.
    - Persistenz: Projektstand wird primär durch das Notebook und die .py-Dateien gesichert. RL-Modelle werden als .zip-Dateien gespeichert. Wichtige Entscheidungen werden manuell oder per Skript in .md/.txt-Dateien festgehalten.

    2. Spiel-Logik (rpg_game_logic.py)
    ----------------------------------
    - Klassenbasiert: Implementierung mit Klassen für Character, Skill und Environment.
    - Charakterattribute: Definierte Basisattribute (STR, GES, KON, INT, WEI) und abgeleitete Kampfstatistiken (Angriff, Verteidigung, Magiekraft etc.). Ressourcen: HP, Mana, Stamina, Energy mit passiver Regeneration.
    - Skills: Skills haben Kosten (Ressourcentyp variabel), Abklingzeiten, Zieltypen, Effekttypen (Schaden, Heilung, Buff, Debuff), Stärke, optional Elemente und Dauer.
    - Effektsystem: Unterstützung für Buffs, Debuffs, Schaden-über-Zeit (DoT) und Heilung-über-Zeit (HoT) über ein active_effects-Dictionary in der Character-Klasse. Berücksichtigt Dauer und (optional) Stacking.
    - XP & Leveling: Helden sammeln XP durch das Besiegen von Gegnern und steigen im Level auf, was Attribute erhöht und Ressourcen wiederherstellt.
    - Kampfablauf (Basis): Rundenbasiert. Charaktere agieren basierend auf Initiative (aktuell GES). Start-of-Turn-Effekte (Cooldowns, DoT/HoT, Regeneration) werden angewendet.

    3. Künstliche Intelligenz (KI)
    -----------------------------
    - Hybrid-Ansatz:
        - Gegner-KI: Skriptbasiert (_scripted_enemy_ai in rpg_env.py). Nutzt einfache Heuristiken (Heilung bei niedriger HP, Debuffs anwenden, Buffs anwenden, stärksten Schadensskill nutzen, sonst Basis-Angriff).
        - Helden-KI: Reinforcement Learning (RL) mit stable-baselines3 (Algorithmus: PPO).
    - Agenten-Strategie: Ein separater PPO-Agent wird für jede Heldenklasse trainiert.

    4. RL-Umgebung (rpg_env.py)
    --------------------------
    - Basis: Implementiert als gymnasium.Env-Subklasse.
    - Action Space: spaces.Discrete. Aktionen umfassen alle definierten Skills sowie dedizierte Aktionen für "Passen" und "Basis-Angriff".
    - Observation Space: spaces.Box (kontinuierlich, normalisiert auf 0-1). Beinhaltet:
        - Normalisierte Ressourcen (HP, Mana, etc.) von Held und Gegner.
        - Normalisierte Skill-Cooldowns des Helden.
        - Status von Buffs/Debuffs (Dauer, Stacks) auf Held und Gegner.
        - Normalisiertes Level und XP-Fortschritt des Helden.
    - reset()-Methode: Initialisiert einen Kampf. Wählt Heldenklasse und Gegnertyp aus (kann für Training/Evaluierung fixiert werden via set_fixed_class/set_fixed_opponent). Setzt Level basierend auf einer leichten Varianz.
    - step()-Methode: Verarbeitet die Aktion des Helden-Agenten, führt die Heldenaktion aus, lässt die Gegner-KI agieren, wendet Effekte an, prüft auf Kampfende (Tod eines Charakters oder Erreichen von MAX_TURNS), berechnet die Belohnung und gibt die nächste Observation zurück. Enthält is_success-Info für Monitoring.
    - Reward Shaping: Umfasst Belohnungen/Strafen für:
        - Sieg/Niederlage/Timeout.
        - Verursachten Schaden (relativ zur max HP des Gegners).
        - Rest-HP bei Sieg.
        - Level-Ups.
        - Erfolgreiche Skill-Nutzung.
        - Anwenden spezifischer nützlicher Buffs/Debuffs.
        - Malus für Passen, Basis-Angriff (Held), ungültige Aktionen, lange Kämpfe, Skill-Fehlversuche.
    - Debugging: Enthält aktuell print-Anweisungen in step() zur Diagnose von Endlosschleifen (sollten nach Behebung entfernt werden).

    5. Benutzeroberfläche (UI) (rpg_ui.py)
    --------------------------------------
    - Technologie: Dynamisch generiertes HTML mit CSS, angezeigt in Colab via display(HTML(...)).
    - Zweck: Primär zur visuellen Beobachtung des Spielzustands und Kampfablaufs.
    - Inhalt: Zeigt Charakterboxen mit HP, Ressourcen, Level/XP, aktiven Effekten und Cooldowns an. Enthält ein farblich hervorgehobenes Kampf-Log.

    6. Training (Zelle 5, rpg_training_utils.py)
    ---------------------------------------------
    - Workflow: Haupt-Trainingsschleife in Zelle 5 iteriert über CLASSES_TO_TRAIN (aus rpg_config.py).
    - Umgebung: Für jeden Trainingslauf wird eine eigene RPGEnv-Instanz erstellt und mit Monitor gewrapped, um Statistiken (Belohnung, Länge, Erfolg) in monitor.csv zu loggen.
    - Modell: PPO-Modell wird pro Klasse erstellt oder geladen (LOAD_EXISTING_MODELS Flag).
    - Fortschritt: Visuelle Fortschrittsanzeige via IPyWidgetProgressCallback (Balken, Zeitangaben).
    - Berichterstellung: Nach jedem Klassentraining wird automatisch ein verständlicher Textbericht (trainingsbericht_*.txt) mittels generate_layperson_log erstellt, der den Trainingsfortschritt zusammenfasst. Berichte werden in täglichen Unterordnern gespeichert.
    - Debugging: Debug-Ausgaben aus rpg_env.py werden während des Trainings in ein separates Output-Widget umgeleitet (konfiguriert in Zelle 5).

    7. Evaluierung (Zelle 7, rpg_training_utils.py)
    ----------------------------------------------
    - Workflow: Haupt-Evaluierungsschleife in Zelle 7 iteriert über CLASSES_TO_TRAIN.
    - Funktion: evaluate_agent_and_summarize führt die Evaluierung pro Klasse durch.
    - Durchführung: Lässt den trainierten Agenten eine feste Anzahl von Kämpfen (N_FIGHTS_PER_OPPONENT) gegen definierte Gegnertypen spielen. Aktionswahl deterministisch oder stochastisch (EVAL_DETERMINISTIC).
    - Datenerfassung: Sammelt detaillierte Statistiken pro Kampf und aggregiert sie (Sieg/Niederlage/Timeout-Raten, Ø Belohnung, Ø Dauer, Ø Rest-Ressourcen, Ø Schaden/Heilung, Skill-Nutzung, Passen/Basis-Angriff-Details). Erfasst Statistiken auch pro Gegnertyp.
    - Logging: Schreibt ein detailliertes Schritt-Log (evaluation_steps_*.csv) mit dem Zustand und den Aktionen in jeder Runde jedes Kampfes.
    - Berichterstellung: Generiert automatisch detaillierte Zusammenfassungen als .txt-Datei (optional auch .csv für Vergleich, .html für Webansicht). Berichte werden in täglichen Unterordnern gespeichert.
    - Ausgabe-Management: Leitet während der Ausführung die allgemeinen Statusmeldungen und das detaillierte Kampf-Log in separate ipywidgets.Output-Widgets um (konfiguriert in Zelle 7), um die Übersichtlichkeit zu verbessern.

    8. Konfiguration (rpg_config.py, rpg_definitions.py)
    ----------------------------------------------------
    - rpg_config.py: Enthält globale Konfigurationen wie Verzeichnispfade, zu trainierende Klassen, Trainings-Timesteps, PPO-Hyperparameter, Evaluierungsparameter (MAX_TURNS, EVAL_DETERMINISTIC).
    - rpg_definitions.py: Definiert die "Datenbank" des Spiels: Basisattribute der Charakterklassen (CHAR_PARAMS), Skill-Parameter (SKILL_PARAMS), Zuordnung von Skills zu Klassen (ALL_SKILLS_MAP), Primärskills (PRIMARY_SKILLS), Hauptressourcen (CLASS_MAIN_RESOURCE), Liste aller Buff-/Debuff-Namen.

    9. Aktuelle Herausforderungen & Nächste Schritte
    ------------------------------------------------
    - Balance: Größte aktuelle Herausforderung. Orks sind zu stark, Skelettmagier zu schwach, Krieger oft dominant. Mehrere Balance-Anpassungen waren bisher nicht ausreichend. Ein radikales Rebalancing ist erforderlich.
    - Debugging: Die Ursache für potenziell unendliche Kämpfe (Endlosschleife in Zelle 5) wird aktuell mittels Debug-Ausgaben in rpg_env.py untersucht.
    - Analyse: Nutzung der generierten Evaluierungsberichte und Schritt-Logs zur genaueren Analyse des Agentenverhaltens (z.B. Kleriker-Fehlversuche, spezifische Kampfverläufe).
    """

        try:
            # Schreibe den Text in die Text-Datei
            with open(filename, "w", encoding="utf-8") as f:
                f.write(summary_text)
            print(f"Entscheidungszusammenfassung erfolgreich in '{filename}' geschrieben.")
        except IOError as e:
            print(f"FEHLER beim Schreiben der Datei '{filename}': {e}")
        except Exception as e:
            print(f"Ein unerwarteter Fehler ist aufgetreten: {e}")

    # --- Skript ausführen ---
    # Erzeugt die Text-Datei, wenn diese Zelle ausgeführt wird
    generate_decision_summary_txt()

    # Optional: Zweite Funktion für eine kürzere Projektzusammenfassung (falls benötigt)
    # def generate_project_summary_txt(filename="Projekt_Zusammenfassung.txt"):
    #     # ... (Ähnliche Logik mit kürzerem Text) ...
    #     pass
    # generate_project_summary_txt()


Generiere detaillierte Entscheidungszusammenfassung nach 'ENTSCHEIDUNGEN_AKTUELL.txt'...
Entscheidungszusammenfassung erfolgreich in 'ENTSCHEIDUNGEN_AKTUELL.txt' geschrieben.


In [ ]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 9.1: Generator für ENTSCHEIDUNGEN.txt
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import datetime
import os
import traceback

print("Aktualisiere ENTSCHEIDUNGEN.txt...")

# Aktuelles Datum und Uhrzeit holen
try:
    now = datetime.datetime.now()
    # Verwende ein Format, das für Dateinamen und Anzeige gut ist
    timestamp_str = now.strftime("%Y-%m-%d %H:%M:%S")
except Exception as e:
    print(f"Fehler beim Holen des Zeitstempels: {e}")
    timestamp_str = "UNBEKANNT"

# --- Der Inhalt der Datei ---
# HINWEIS: Du kannst diesen Textblock jederzeit anpassen, um den aktuellen Stand widerzuspiegeln!
# Es ist sinnvoll, hier die wichtigsten Punkte aus der obigen Markdown-Zusammenfassung einzufügen.
content = f"""
# Projektentscheidungen & Status: Python RPG (Stand: {timestamp_str})

Dieses Dokument fasst wichtige grundlegende und aktuelle Design- und Implementierungsentscheidungen sowie den aktuellen Status des Projekts zusammen.

## 1. Grundlegende Projektdefinition

* **Ziel:** Selbstlaufendes RPG in Python zur Beobachtung von KI-Verhalten.
* **Umgebung:** Python, Google Colab, `.py`-Module, Haupt-Notebook (`RPG.ipynb`).
* **UI:** Dynamisches HTML in Colab (`rpg_ui.py`) zur Beobachtung.
* **KI-Strategie:** Skript-KI für Gegner (Regel-basiert V2.2), RL (PPO) für Helden.
* **Kampfmodus:** Helden treten nur gegen dedizierte Gegnertypen an.

## 2. Wichtige Entwicklungen / Kern-Entscheidungen (Auswahl)

* Basis-System, RL-Agenten, Spielmechanik, Reward Shaping implementiert.
* Diverse Debugging-Phasen durchlaufen (Imports, Reloads, Logik).
* Gegner-Strategie angepasst (kein Held vs. Held).
* Gegner-Pool erweitert (Goblin Varianten, Ork Berserker, Skelett Magier).
* Gegner-KI verbessert (Buff/Debuff/Heal/BA-Logik).
* Gegner-Aktions-Tracking und Basis-Angriff implementiert.
* Mehrere Balance-Anpassungen durchgeführt (V14, V15, V16) - bisher wenig erfolgreich.
* Detaillierte Evaluierungsberichte und Schritt-Logs implementiert.

## 3. Aktueller Stand & Analyse (Basierend auf Eval mit Defs V15 / Env V22)

* **KI Funktionalität:** Gegner-KI (V2.2) und Helden-KI (PPO) laufen technisch.
* **Helden-Performance (Kurz):** Krieger (dominant), Kleriker (gut, aber Fehler/vs Ork), Schurke/Magier (mittel, Probleme vs Ork).
* **Gegner-Balance:** Orks (Basis/Berserker) zu stark. Skelett Magier zu schwach. Goblins OK.
* **Gesamtbalance:** Schieflage durch Orks und Krieger-Dominanz.
* **Offene Punkte:** Ork-Übermacht, Skelett-Magier-Schwäche, Krieger-Dominanz, Kleriker-Fehlversuche analysieren, N/A-Aktionen im Log prüfen.

## 4. Nächste Schritte (Fokus)

* **Priorität:** Radikales Rebalancing (Orks, Skelett Magier, Krieger) in `rpg_definitions.py`.
* **Analyse:** Nutzung der Schritt-Logs (`evaluation_steps_*.csv`) zur Detailanalyse (Kleriker-Fehler, Ork-Kämpfe).
* **Code-Bereinigung:** Redundante Funktionsdefinition in Zelle 5 entfernen. Sicherstellen, dass `N_FIGHTS_PER_OPPONENT` korrekt in `rpg_training_utils.py` definiert ist (V24 empfohlen).
* **Optional:** Berichtsformate aus Evaluierung anpassen (Multi-Format V18/V19 oder bei V23 bleiben).

## 5. Offene Punkte (Generell)

* Langfristige Erweiterungen (Items, Quests etc.) zurückgestellt.
"""

# Schreibe den Inhalt in die Datei
# Stelle sicher, dass der Pfad korrekt ist (aus Zelle 1/2)
try:
    # Prüfe, ob project_path existiert und ein Verzeichnis ist
    if 'project_path' in locals() and isinstance(project_path, str) and os.path.isdir(project_path):
        pass # Alles gut
    elif 'project_path' in globals() and isinstance(globals()['project_path'], str) and os.path.isdir(globals()['project_path']):
         project_path = globals()['project_path'] # Hole aus globalem Scope
    else:
        # Versuche, den Pfad aus rpg_config zu holen, falls Zelle 2 nicht lief, aber Config geladen wurde
        if 'rpg_config' in locals() and hasattr(rpg_config, 'BASE_PROJECT_PATH'):
            project_path = rpg_config.BASE_PROJECT_PATH
            print(f"WARNUNG: 'project_path' nicht gefunden, verwende BASE_PROJECT_PATH aus rpg_config: {project_path}")
            if not os.path.isdir(project_path):
                 print(f"WARNUNG: Fallback-Pfad '{project_path}' ist auch ungültig. Verwende aktuelles Verzeichnis.")
                 project_path = os.getcwd()
        else:
            print("WARNUNG: Variable 'project_path' nicht gefunden oder ungültig. Verwende aktuelles Verzeichnis.")
            project_path = os.getcwd()
except NameError:
     print("WARNUNG: Variable 'project_path' nicht definiert. Verwende aktuelles Verzeichnis.")
     project_path = os.getcwd()

# Definiere den Dateinamen als .txt
file_name = "ENTSCHEIDUNGEN.txt"
file_path = os.path.join(project_path, file_name)
print(f"Versuche in Datei zu schreiben: {file_path}")

try:
    # Stelle sicher, dass das Zielverzeichnis existiert
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, "w", encoding="utf-8") as f:
        # Entferne führende/trailing Leerzeichen/Zeilenumbrüche vom content
        f.write(content.strip() + "\n") # Füge einen einzelnen Zeilenumbruch am Ende hinzu
    print(f"Datei '{file_name}' erfolgreich unter '{file_path}' geschrieben/aktualisiert.")
except Exception as e:
    print(f"FEHLER beim Schreiben der Datei '{file_name}': {e}")
    print(f"Versuchter Pfad: {file_path}")
    traceback.print_exc()



Aktualisiere ENTSCHEIDUNGEN.txt...
Versuche in Datei zu schreiben: /content/drive/MyDrive/Colab Notebooks/ENTSCHEIDUNGEN.txt
Datei 'ENTSCHEIDUNGEN.txt' erfolgreich unter '/content/drive/MyDrive/Colab Notebooks/ENTSCHEIDUNGEN.txt' geschrieben/aktualisiert.


In [ ]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE 9.3: Backup von .ipynb und .py Dateien als .txt
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import os
import shutil
import datetime
import traceback

print("Starte Backup-Prozess für .ipynb und .py Dateien als .txt...")

# --- Konfiguration ---
# Stelle sicher, dass der Projektpfad korrekt ist (aus Zelle 1/2)
if 'project_path' not in locals() or not os.path.isdir(project_path):
     print("WARNUNG: 'project_path' nicht korrekt gesetzt oder Verzeichnis existiert nicht.")
     project_path = os.getcwd()
     print(f"Verwende aktuelles Verzeichnis als Fallback: {project_path}")

# Name des Notebooks (ggf. anpassen)
notebook_filename = "RPG.ipynb" # Passe dies an, falls dein Notebook anders heißt

# Liste der Python-Module (Namen ohne .py)
# Du kannst diese Liste manuell pflegen oder versuchen, sie dynamisch zu bekommen
# Hier verwenden wir die Liste, die wir kennen:
module_names = [
    'rpg_config',
    'rpg_definitions',
    'rpg_game_logic',
    'rpg_ui',
    'rpg_env',
    'rpg_training_utils'
]

# Verzeichnis für die Text-Backups erstellen
backup_dir_name = "backup_txt"
backup_dir_path = os.path.join(project_path, backup_dir_name)
try:
    os.makedirs(backup_dir_path, exist_ok=True)
    print(f"Backup-Verzeichnis sichergestellt: {backup_dir_path}")
except Exception as e:
    print(f"FEHLER beim Erstellen des Backup-Verzeichnisses: {e}")
    # Hier vielleicht abbrechen, wenn das Verzeichnis nicht erstellt werden kann?
    # raise e # würde die Zelle stoppen

# --- Dateien zum Sichern zusammenstellen ---
files_to_backup = []
# Notebook hinzufügen
notebook_path = os.path.join(project_path, notebook_filename)
if os.path.exists(notebook_path):
    files_to_backup.append(notebook_filename)
else:
    print(f"WARNUNG: Notebook-Datei '{notebook_filename}' nicht im Projektpfad gefunden.")

# Python-Dateien hinzufügen
for module_name in module_names:
    py_filename = module_name + ".py"
    py_path = os.path.join(project_path, py_filename)
    if os.path.exists(py_path):
        files_to_backup.append(py_filename)
    else:
        print(f"WARNUNG: Python-Datei '{py_filename}' nicht im Projektpfad gefunden.")

print(f"\nFolgende Dateien werden als .txt gesichert (falls vorhanden): {files_to_backup}")

# --- Backup-Vorgang ---
backup_count = 0
error_count = 0
for filename in files_to_backup:
    source_path = os.path.join(project_path, filename)
    # Backup-Dateiname: original.py -> original.py.txt oder original.ipynb -> original.ipynb.txt
    backup_filename = filename + ".txt"
    dest_path = os.path.join(backup_dir_path, backup_filename)

    try:
        print(f"  Sichere '{filename}' nach '{backup_filename}'...")
        # Kopiere den Inhalt Zeile für Zeile (sicherer für verschiedene Encodings)
        with open(source_path, 'r', encoding='utf-8', errors='ignore') as f_source, \
             open(dest_path, 'w', encoding='utf-8') as f_dest:
            for line in f_source:
                f_dest.write(line)
        backup_count += 1
        print(f"    -> Erfolgreich.")
    except FileNotFoundError:
        print(f"    -> FEHLER: Quelldatei '{filename}' nicht gefunden (sollte nicht passieren).")
        error_count += 1
    except Exception as e:
        print(f"    -> FEHLER beim Sichern von '{filename}': {e}")
        traceback.print_exc()
        error_count += 1

print(f"\nBackup-Prozess abgeschlossen.")
print(f"- Erfolgreich gesichert: {backup_count} Datei(en)")
if error_count > 0:
    print(f"- Fehler aufgetreten: {error_count} Datei(en)")
print(f"- Backups gespeichert in: {backup_dir_path}")

Starte Backup-Prozess für .ipynb und .py Dateien als .txt...
Backup-Verzeichnis sichergestellt: /content/drive/MyDrive/Colab Notebooks/backup_txt

Folgende Dateien werden als .txt gesichert (falls vorhanden): ['RPG.ipynb', 'rpg_config.py', 'rpg_definitions.py', 'rpg_game_logic.py', 'rpg_ui.py', 'rpg_env.py', 'rpg_training_utils.py']
  Sichere 'RPG.ipynb' nach 'RPG.ipynb.txt'...
    -> Erfolgreich.
  Sichere 'rpg_config.py' nach 'rpg_config.py.txt'...
    -> Erfolgreich.
  Sichere 'rpg_definitions.py' nach 'rpg_definitions.py.txt'...
    -> Erfolgreich.
  Sichere 'rpg_game_logic.py' nach 'rpg_game_logic.py.txt'...
    -> Erfolgreich.
  Sichere 'rpg_ui.py' nach 'rpg_ui.py.txt'...
    -> Erfolgreich.
  Sichere 'rpg_env.py' nach 'rpg_env.py.txt'...
    -> Erfolgreich.
  Sichere 'rpg_training_utils.py' nach 'rpg_training_utils.py.txt'...
    -> Erfolgreich.

Backup-Prozess abgeschlossen.
- Erfolgreich gesichert: 7 Datei(en)
- Backups gespeichert in: /content/drive/MyDrive/Colab Notebooks/ba

## 10: Manueller Test für XP & Level Up

In [ ]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ZELLE (z.B. 10): Manueller Test für XP & Level Up (inkl. manuellem LvlUp-Check)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import importlib
import time
import traceback # Für detailliertere Fehlermeldungen
from IPython.display import display, clear_output
import random

# --- Sicherstellen, dass die UI Funktion existiert ---
# (Normalerweise durch Zelle 2 importiert, aber zur Sicherheit hier prüfen)
if 'rpg_ui' not in locals() or not hasattr(rpg_ui, 'generate_html_ui'):
     print("FEHLER: UI-Funktion 'generate_html_ui' nicht gefunden. Bitte Zelle 2 ausführen.")
     # Hier besser abbrechen
     raise NameError("UI-Funktion nicht verfügbar.")


print("--- Starte Manuellen Testlauf ---")

# --- 1. Module neu laden (WICHTIG!) ---
# Lädt die neuesten Versionen der .py Dateien
try:
    # Stelle sicher, dass die Variablen aus vorherigen Zellen existieren
    if 'rpg_definitions' not in locals(): raise NameError("Modul 'rpg_definitions' nicht importiert.")
    if 'rpg_game_logic' not in locals(): raise NameError("Modul 'rpg_game_logic' nicht importiert.")
    if 'rpg_env' not in locals(): raise NameError("Modul 'rpg_env' nicht importiert.")
    if 'rpg_ui' not in locals(): raise NameError("Modul 'rpg_ui' nicht importiert.")

    importlib.reload(rpg_definitions)
    importlib.reload(rpg_game_logic)
    importlib.reload(rpg_env)
    importlib.reload(rpg_ui)
    print("Module rpg_definitions, rpg_game_logic, rpg_env, rpg_ui neu geladen.")
except NameError as e:
    print(f"FEHLER: Konnte Module nicht neu laden, da sie nicht importiert wurden. Stelle sicher, dass Zelle 2/3 gelaufen sind. Fehler: {e}")
    raise e
except Exception as e:
    print(f"FEHLER beim Neuladen der Module: {e}")
    traceback.print_exc()
    raise e

# --- 2. Umgebung holen oder neu erstellen ---
env = None # Sicherstellen, dass env definiert ist
# Versuche, die bestehende Instanz aus Zelle 4 zu verwenden
if 'rpg_environment_instance' in locals() and isinstance(rpg_environment_instance, rpg_env.RPGEnv):
    # Prüfen ob die Instanz mit der neu geladenen Klasse übereinstimmt
    if type(rpg_environment_instance) == rpg_env.RPGEnv:
        env = rpg_environment_instance
        print("Verwende bestehende RPGEnv Instanz.")
    else:
        print("Bestehende Instanz ist vom alten Typ, erstelle neue Instanz...")
        env = None # Erzwinge Neuerstellung

if env is None:
    print("Erstelle neue RPGEnv Instanz für den Test...")
    try:
        # Sicherstellen, dass INITIALIZED_SKILLS aus Zelle 3 existiert
        if 'INITIALIZED_SKILLS' not in locals():
            raise NameError("INITIALIZED_SKILLS nicht gefunden. Bitte Zelle 3 ausführen.")

        env = rpg_env.RPGEnv(
            char_param_definitions=rpg_definitions.CHAR_PARAMS,
            all_buffs_debuffs_names=rpg_definitions.ALL_BUFFS_DEBUFFS_NAMES,
            max_buff_duration=rpg_definitions.MAX_BUFF_DURATION,
            max_stacks=rpg_definitions.MAX_STACKS,
            render_mode='ansi'
        )
        rpg_environment_instance = env # Speichere die neue Instanz global, falls sie wiederverwendet wird
        print("Neue RPGEnv Instanz erstellt und als 'rpg_environment_instance' gespeichert.")
    except Exception as e:
        print(f"FEHLER beim Erstellen der Test-RPGEnv Instanz: {e}")
        traceback.print_exc()
        raise e

# --- 3. Testparameter ---
hero_test_class = "Krieger" # Klasse zum Testen
# enemy_test_class = "Schurke" # Wird jetzt durch Env bestimmt
max_test_turns = 30       # Maximale Runden für diesen Test (ggf. erhöhen/senken)
# Wähle eine Aktion zum Testen (Basis-Angriff ist zuverlässig)
action_to_test = env._basic_attack_idx
action_name_to_test = env.action_to_name_map.get(action_to_test, "Unbekannt")

print(f"\nTest Setup: Held={hero_test_class}, Aktion={action_name_to_test} ({action_to_test}), Max Runden={max_test_turns}")

# --- 4. Testdurchführung ---
log_history = []
try:
    # Umgebung mit fester Heldenklasse zurücksetzen
    env.set_fixed_class(hero_test_class)
    obs, info = env.reset()

    # Überprüfe, ob Held und Gegner erstellt wurden
    if not env.hero or not env.enemy:
         raise RuntimeError("Held oder Gegner konnte im Reset nicht erstellt werden.")

    log_history.append(f"Testkampf beginnt: {env.hero.name} (Lvl {env.hero.level}) vs {env.enemy.name} (Lvl {env.enemy.level})")
    print(f"Startzustand: Held={str(env.hero)}")
    print(f"Startzustand: Gegner={str(env.enemy)}")

    # UI initial anzeigen
    clear_output(wait=True)
    display(rpg_ui.generate_html_ui(env, log_history))
    print(f"Runde 0: Kampf beginnt.")
    # time.sleep(1) # Kurze Pause zu Beginn

    terminated = False
    truncated = False
    turn = 0

    while not terminated and not truncated and turn < max_test_turns:
        turn += 1
        #print(f"\n--- Runde {turn} ---") # Weniger Output während des Laufs

        # Feste Aktion ausführen
        current_action = action_to_test

        # Umgebungsschritt
        obs, reward, terminated, truncated, info = env.step(current_action)

        # Log-Nachrichten hinzufügen (selektiv)
        action_name = env.action_to_name_map.get(current_action, "Unbekannt")
        log_entry = f"R {turn}: {env.hero.name} nutzt {action_name}."
        if info.get('last_reward_components', {}).get('Schaden verursacht Bonus', 0) > 0 :
             # Finde den tatsächlichen Schaden (ungefähre Schätzung aus Reward)
             # hp_diff = info['last_reward_components']['Schaden verursacht Bonus'] / env.DAMAGE_REWARD_FACTOR * env.enemy.max_hp
             # log_entry += f" ({env.enemy.name} erleidet Schaden)" # Vereinfacht
             pass # Weniger ausführlich im Log
        log_history.append(log_entry)

        # Wichtige Ereignisse loggen
        if info.get('last_reward_components', {}).get('Sieg', 0) > 0:
            log_history.append(f"R {turn}: {env.enemy.name} wurde besiegt!")
            print(f"Runde {turn}: Gegner besiegt!")
        if info.get('last_reward_components', {}).get('Niederlage', 0) < 0:
             log_history.append(f"R {turn}: {env.hero.name} wurde besiegt!")
             print(f"Runde {turn}: Held besiegt!")
        if info.get('last_reward_components', {}).get('Level Up Bonus', 0) > 0:
             level_up_msg = f"R {turn}: LEVEL UP für {env.hero.name} auf Level {env.hero.level}!"
             log_history.append(level_up_msg)
             print(level_up_msg)


        # Aktuellen Zustand und UI anzeigen (weniger häufig für Übersicht)
        if turn % 1 == 0 or terminated or truncated: # Update jede Runde oder am Ende
            clear_output(wait=True) # Löscht vorherige Ausgabe
            display(rpg_ui.generate_html_ui(env, log_history[-20:])) # Zeige die letzten 20 Log-Einträge
            print(f"Runde {turn}: Aktion={action_name}, Reward={reward:.2f}, Term={terminated}, Trunc={truncated}")
            # print(f"Info: {info}") # Info ist jetzt sehr lang, ggf. weglassen
            print(f"Held:  {str(env.hero)}")
            print(f"Gegner:{str(env.enemy)}")
            time.sleep(0.2) # Kurze Pause für die Anzeige

    # --- 5. Ergebnis prüfen ---
    print("\n--- Testlauf Kampfphase beendet ---")
    if env.hero:
        print(f"Finaler Heldenstatus nach Kampf: {str(env.hero)}")
        if terminated and not env.hero.is_dead() and hasattr(env.enemy, '_xp_granted') and env.enemy._xp_granted:
             print("Info: Gegner besiegt, XP sollten vergeben worden sein.")
        elif not terminated:
             print("Info: Kampf nicht normal beendet (Timeout/Max Turns).")

    if terminated:
        print("Kampf normal beendet.")
    if truncated:
        print("Kampf durch Timeout beendet.")
    if turn >= max_test_turns and not (terminated or truncated):
         print(f"Kampf nach maximalen Testrunden ({max_test_turns}) beendet.")

    # --- 6. Optionaler manueller Level-Up Test ---
    if env.hero and not env.hero.is_dead():
        print("\n--- Manueller Level-Up Test ---")
        # Berechne benötigte XP für nächsten Level Up
        xp_needed_for_next = env.hero.xp_to_next_level - env.hero.current_xp
        # Gebe genug XP + 1, um sicher aufzusteigen
        xp_to_add = max(1, xp_needed_for_next + 1)

        print(f"Aktuell: Lvl {env.hero.level}, XP {env.hero.current_xp}/{env.hero.xp_to_next_level}")
        print(f"Gebe {env.hero.name} manuell {xp_to_add} XP...")
        level_before_manual_xp = env.hero.level
        env.hero.gain_xp(xp_to_add) # Füge XP hinzu

        # Prüfe ob Level Up stattgefunden hat
        if env.hero.level > level_before_manual_xp:
            print(f"ERFOLG: Manueller Level Up hat stattgefunden! Neuer Level: {env.hero.level}")
            print(f"Neuer Status nach manuellem XP-Push: {str(env.hero)}")
            # Optional: Prüfe, ob Stats sich sinnvoll geändert haben (visuell)
        else:
            print("WARNUNG: Manueller Level Up hat NICHT stattgefunden? Überprüfe XP-Logik oder ob Held schon Max-Level ist.")
            print(f"Status nach manuellem XP-Push: {str(env.hero)}")
    elif env.hero and env.hero.is_dead():
         print("\nManueller Level-Up Test übersprungen (Held ist tot).")
    else:
        print("\nManueller Level-Up Test übersprungen (Held nicht vorhanden).")


except Exception as e:
    print(f"\nFEHLER während des Testlaufs: {e}")
    traceback.print_exc()

print("\n--- Testlauf Skript Ende ---")

Runde 7: Aktion=Basis-Angriff, Reward=19.27, Term=True, Trunc=False
Held:  Krieger_Hero (Krieger L:1) HP:135/145 XP:63/100 S:100/100 Atr:[S:15 G:8 K:12 I:5 W:5]
Gegner:Skelett_L2_979 (Skelett L:2) HP:0/70 M:30/30 Atr:[S:6 G:8 K:4 I:8 W:6]

--- Testlauf Kampfphase beendet ---
Finaler Heldenstatus nach Kampf: Krieger_Hero (Krieger L:1) HP:135/145 XP:63/100 S:100/100 Atr:[S:15 G:8 K:12 I:5 W:5]
Info: Gegner besiegt, XP sollten vergeben worden sein.
Kampf normal beendet.

--- Manueller Level-Up Test ---
Aktuell: Lvl 1, XP 63/100
Gebe Krieger_Hero manuell 38 XP...
LEVEL UP! Krieger_Hero ist jetzt Level 2!
Attribute erhöht: KON +1, STR +1
Krieger_Hero regeneriert beim Level Up (+15 HP).
Krieger_Hero Ressourcen aufgefüllt.
ERFOLG: Manueller Level Up hat stattgefunden! Neuer Level: 2
Neuer Status nach manuellem XP-Push: Krieger_Hero (Krieger L:2) HP:150/160 XP:1/200 S:100/100 Atr:[S:16 G:8 K:13 I:5 W:5]

--- Testlauf Skript Ende ---


# Code

In [ ]:
%%writefile rpg_training_utils.py
# -*- coding: utf-8 -*-
# Generated/Overwritten on: 2025-04-17 22:08:00 # Zeitpunkt aktualisiert
# rpg_training_utils.py
# V26: Removed redundant Character/Skill assignments inside evaluate_agent_and_summarize try-block.

import os, sys, time, datetime, math, traceback, importlib, inspect
import numpy as np
import pandas as pd
import csv
from collections import defaultdict
from typing import Optional, Dict, List, Any, Set, Tuple
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results
import html

import ipywidgets as widgets
from IPython.display import display

# Globale Imports - Diese definieren Character und Skill global für dieses Modul
try:
    from rpg_env import RPGEnv
    import rpg_game_logic
    Character = rpg_game_logic.Character # Explizit für Type Hinting & globalen Zugriff
    Skill = rpg_game_logic.Skill         # Explizit für Type Hinting & globalen Zugriff
    import rpg_config
    import rpg_definitions
    print("INFO (rpg_training_utils V26): Kernmodule erfolgreich importiert.")
except ImportError as e:
    print(f"FEHLER (rpg_training_utils V26): Globale Modul-Imports fehlgeschlagen! Definiere Dummies. FEHLER: {e}")
    class RPGEnv: pass
    class Character:
        def can_use_skill(self, skill): return False
    class Skill: pass
    class rpg_config: DEFAULT_PPO_PARAMS={'policy': "MlpPolicy", 'verbose': 0}; MAX_TURNS=100; MODEL_DIR_BASE="./models_fallback/"; SUMMARY_SAVE_DIR="./summaries_fallback/"; LOG_DIR_BASE="./logs_fallback/"
    class rpg_definitions: CHAR_PARAMS={}; ALL_SKILLS_MAP={}; PRIMARY_SKILLS={}; CLASS_MAIN_RESOURCE={}; ALL_BUFFS_DEBUFFS_NAMES=[]; MAX_BUFF_DURATION=5; MAX_STACKS=3

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Callback mit IPyWidgets für Fortschrittsanzeige (Unverändert zu V24)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ... (Code für IPyWidgetProgressCallback unverändert) ...
class IPyWidgetProgressCallback(BaseCallback):
    """ Custom callback for progress display using IPyWidgets. (Unchanged) """
    def __init__(self, total_timesteps: int, verbose: int = 0):
         super().__init__(verbose); self.target_total_timesteps = total_timesteps; self.progress_bar=None; self.step_label=None; self.time_label=None; self.widget_container=None; self.start_time=0; self.initial_model_num_timesteps=0; self.update_freq=max(100,total_timesteps//100)
    def _on_training_start(self) -> None:
        self.initial_model_num_timesteps=self.num_timesteps; self.start_time=time.time(); max_val=float(max(1.0,self.target_total_timesteps)); self.progress_bar=widgets.FloatProgress(value=float(self.initial_model_num_timesteps),min=0.0,max=max_val,description='Training:',bar_style='info',orientation='horizontal',layout=widgets.Layout(width='70%')); self.step_label=widgets.Label(value=f"{self.initial_model_num_timesteps}/{self.target_total_timesteps} Steps",layout=widgets.Layout(margin='0 0 0 10px')); self.time_label=widgets.Label(value="Dauer: 00:00:00 | Restzeit: ??:??:??"); progress_line=widgets.HBox([self.progress_bar,self.step_label]); self.widget_container=widgets.VBox([progress_line,self.time_label]); display(self.widget_container)
    def _on_step(self) -> bool:
        if self.n_calls%self.update_freq==0:
            if self.progress_bar and self.step_label and self.time_label:
                current_steps=self.num_timesteps; self.progress_bar.max=float(max(self.target_total_timesteps,current_steps)); self.progress_bar.value=float(current_steps); self.step_label.value=f"{current_steps}/{self.target_total_timesteps} Steps"; elapsed_time=time.time()-self.start_time; steps_done_in_run=current_steps-self.initial_model_num_timesteps; remaining_steps=max(0,self.target_total_timesteps-current_steps); estimated_remaining_time=float('inf')
                if steps_done_in_run>self.update_freq and elapsed_time>1: time_per_step=elapsed_time/steps_done_in_run; estimated_remaining_time=remaining_steps*time_per_step
                elapsed_str=time.strftime("%H:%M:%S",time.gmtime(elapsed_time)); remaining_str=time.strftime("%H:%M:%S",time.gmtime(estimated_remaining_time)) if estimated_remaining_time!=float('inf') else "??:??:??"; self.time_label.value=f"Dauer: {elapsed_str} | Restzeit: {remaining_str}"
                if self.progress_bar.bar_style=='info' and current_steps>self.initial_model_num_timesteps: self.progress_bar.bar_style='success'
        return True
    def _on_training_end(self) -> None:
         if self.progress_bar and self.step_label and self.time_label:
            final_steps=self.num_timesteps; self.progress_bar.value=float(final_steps); self.step_label.value=f"{final_steps}/{self.target_total_timesteps} Steps"; elapsed_time=time.time()-self.start_time; elapsed_str=time.strftime("%H:%M:%S",time.gmtime(elapsed_time)); self.time_label.value=f"Dauer: {elapsed_str} | Abgeschlossen."; self.progress_bar.bar_style='success' if final_steps>=self.target_total_timesteps else 'warning'

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Funktion zum Trainieren eines Agenten für eine Klasse (Unverändert zu V24)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ... (Code für train_agent_for_class unverändert) ...
def train_agent_for_class(
        env_instance: RPGEnv, class_name: str, total_timesteps: int,
        log_dir_base: str, model_dir_base: str, load_existing: bool = True,
        ppo_params: Optional[dict] = None, progress_callback_cls: type = IPyWidgetProgressCallback
        ) -> Optional[PPO]:
    """ Trains a PPO agent for a specific hero class. (Unchanged) """
    print(f"\n--- Setup Training für Klasse: {class_name} ---"); os.makedirs(log_dir_base, exist_ok=True); os.makedirs(model_dir_base, exist_ok=True)
    timestamp=datetime.datetime.now().strftime("%Y%m%d_%H%M%S"); model_fname_base=f"ppo_{class_name.lower()}_agent"; latest_model_path=os.path.join(model_dir_base,f"{model_fname_base}.zip"); save_model_path_timestamp=os.path.join(model_dir_base,f"{model_fname_base}_{timestamp}.zip")
    model=None; current_steps=0
    try: env_instance.set_fixed_class(class_name); print(f"Umgebung für Klasse '{class_name}' eingestellt.")
    except Exception as e: print(f"FEHLER beim Einstellen der Klasse '{class_name}': {e}"); return None
    if load_existing and os.path.exists(latest_model_path):
        try: print(f"Lade existierendes Modell: {latest_model_path}"); model=PPO.load(latest_model_path,env=env_instance); current_steps=getattr(model,'num_timesteps',0); print(f"  Modell geladen. Bisherige Schritte: {current_steps}")
        except Exception as e: print(f"FEHLER beim Laden {latest_model_path}: {e}. Erstelle neues Modell."); model=None; current_steps=0
    if model is None:
        print("Erstelle neues PPO-Modell.")
        if 'rpg_config' not in sys.modules: import rpg_config
        params_to_use = (ppo_params or rpg_config.DEFAULT_PPO_PARAMS).copy(); params_to_use.pop('tensorboard_log', None) # Remove tensorboard_log if present
        try:
            if 'policy' not in params_to_use: print("FEHLER: 'policy' fehlt! Verwende Fallback 'MlpPolicy'."); params_to_use['policy'] = params_to_use.get('policy', "MlpPolicy")
            model=PPO(env=env_instance,**params_to_use); current_steps=0
            print(f"  Neues Modell erstellt mit Policy: {params_to_use.get('policy', 'Unbekannt')}")
        except Exception as e: print(f"FATALER FEHLER beim Erstellen des PPO-Modells: {e}"); traceback.print_exc(); return None
    if current_steps>=total_timesteps: print(f"Ziel-Timesteps ({total_timesteps}) bereits erreicht ({current_steps}). Überspringe Training.")
    else:
        print(f"Starte Training von {current_steps} bis {total_timesteps}..."); callback_instance=None
        if progress_callback_cls:
             try: callback_instance=progress_callback_cls(total_timesteps=total_timesteps)
             except Exception as e: print(f"WARNUNG: Konnte Progress Callback nicht erstellen: {e}")
        remaining_timesteps=total_timesteps-current_steps
        try:
            learn_params = {'total_timesteps': remaining_timesteps, 'reset_num_timesteps': False, 'log_interval': 10, 'callback': callback_instance, 'progress_bar': False}
            model.learn(**learn_params); print("\nTraining abgeschlossen.")
        except Exception as e: print(f"\nFEHLER während des Trainings für Klasse {class_name}: {e}"); traceback.print_exc()
        finally:
             latest_path = latest_model_path
             try: print(f"Speichere Modell mit Zeitstempel: {save_model_path_timestamp}"); model.save(save_model_path_timestamp); print("  Timestamp-Modell gespeichert.")
             except Exception as e: print(f"\nFEHLER beim Speichern (Timestamp) für {class_name}: {e}")
             try: print(f"Speichere/Überschreibe neuestes Modell: {latest_path}"); model.save(latest_path); print("  Neuestes Modell gespeichert.")
             except Exception as e: print(f"\nFEHLER beim Speichern (Latest) für {class_name}: {e}")
    print(f"--- Training für Klasse: {class_name} beendet ---"); return model


# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Funktion zum Generieren des verständlichen Trainingsberichts (Unverändert zu V24)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# ... (Code für generate_layperson_log unverändert) ...
def generate_layperson_log(
    log_dir: str, output_filepath: str, class_name: str, total_timesteps: int,
    config_params: dict = None, intervals: list = None, window_size: int = 100
    ):
    """
    Generiert einen leicht verständlichen Textbericht über den Trainingsfortschritt
    basierend auf den Daten einer monitor.csv-Datei.
    """
    # (Code der Funktion wie in V23/V24)
    print(f"Versuche, verständlichen Trainingsbericht für '{class_name}' zu erstellen...")
    print(f"Lese Monitor-Daten aus: {log_dir}")
    print(f"Speichere Bericht nach: {output_filepath}")
    monitor_file_path = os.path.join(log_dir, "monitor.csv")
    if not os.path.exists(monitor_file_path): #... (Restlicher Code der Funktion) ...
        print(f"FEHLER: Monitor-Datei nicht gefunden: {monitor_file_path}")
        report_lines = [f"# FEHLER Trainingsbericht: {class_name}-Agent", f"Monitor-Datei nicht gefunden: {monitor_file_path}"]
        try:
            os.makedirs(os.path.dirname(output_filepath), exist_ok=True)
            with open(output_filepath, "w", encoding="utf-8") as f: f.write("\n".join(report_lines))
        except Exception as e_save: print(f"FEHLER beim Speichern der Fehlermeldung: {e_save}")
        return
    try:
        data = pd.read_csv(monitor_file_path, skiprows=1)
        if data.empty: # ... (Warnungsbehandlung) ...
            print(f"WARNUNG: Monitor-Datei {monitor_file_path} ist leer.")
            report_lines = [f"# WARNUNG Trainingsbericht: {class_name}-Agent", f"Monitor-Datei ist leer: {monitor_file_path}"]
            try:
                os.makedirs(os.path.dirname(output_filepath), exist_ok=True)
                with open(output_filepath, "w", encoding="utf-8") as f: f.write("\n".join(report_lines))
            except Exception as e_save: print(f"FEHLER beim Speichern der Warnung: {e_save}")
            return

        data.columns = [col.strip('# ').strip() for col in data.columns] # Added strip() for safety
        report_lines = []; now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        report_lines.append(f"# Trainingsbericht: {class_name}-Agent"); report_lines.append("") # ... (Header Aufbau) ...
        report_lines.append(f"**Ziel:** Ein KI-{class_name} soll lernen, Kämpfe möglichst gut zu bestehen.")
        report_lines.append(f"**Trainingsdauer:** {total_timesteps} Lernschritte (Ziel)")
        report_lines.append(f"**Bericht vom:** {now}")
        if config_params: report_lines.append("**Trainings-Parameter (Auszug):**"); lr = config_params.get('learning_rate', 'N/A'); bs = config_params.get('batch_size', 'N/A'); gamma = config_params.get('gamma', 'N/A'); report_lines.append(f"  - Lernrate: {lr}"); report_lines.append(f"  - Batch Größe: {bs}"); report_lines.append(f"  - Gamma (Discount): {gamma}")
        report_lines.append("---"); report_lines.append("**Wie liest man das?**"); report_lines.append(f"* **Avg Belohnung (Trend):** Zeigt, wie gut der Agent *im Durchschnitt* pro Kampf abschneidet (höher ist besser). Trend über letzte ~{window_size} Kämpfe."); report_lines.append(f"* **Gewinnrate (%):** Prozent gewonnener Kämpfe (letzte ~{window_size}). Benötigt 'is_success'."); report_lines.append(f"* **Kampfdauer (Trend):** Durchschnittliche Rundenzahl pro Kampf (letzte ~{window_size})."); report_lines.append(f"* **Strategie:** Qualitative Einschätzung."); report_lines.append("---"); report_lines.append("**Beobachtungen während des Trainings:**"); report_lines.append("")

        if intervals is None: intervals = [int(p * total_timesteps) for p in [0.1, 0.25, 0.5, 0.75, 1.0]]; intervals = sorted([i for i in intervals if i > 0]);
        if total_timesteps not in intervals: intervals.append(total_timesteps); intervals = sorted(list(set(i for i in intervals if i <= total_timesteps)))

        required_cols = ['r', 'l', 't']
        missing_cols = [col for col in required_cols if col not in data.columns]
        if missing_cols: raise KeyError(f"Fehlende Spalten in monitor.csv: {', '.join(missing_cols)}")

        try:
            data['r'] = pd.to_numeric(data['r'], errors='coerce')
            data['l'] = pd.to_numeric(data['l'], errors='coerce')
            data['t'] = pd.to_numeric(data['t'], errors='coerce')
            data.dropna(subset=['r', 'l', 't'], inplace=True)
            if data.empty:
                 print("WARNUNG: Keine gültigen numerischen Daten in monitor.csv nach Bereinigung.")
                 report_lines.append("WARNUNG: Keine gültigen Datenpunkte in der Monitor-Datei gefunden.")
                 final_report = "\n".join(report_lines); os.makedirs(os.path.dirname(output_filepath), exist_ok=True)
                 with open(output_filepath, "w", encoding="utf-8") as f: f.write(final_report); return
            cumulative_timesteps = data['t']
        except Exception as e: print(f"FEHLER bei Konvertierung/Berechnung kumulativer Timesteps: {e}"); raise e

        last_avg_reward = -float('inf'); reported_intervals = 0
        for step_threshold in intervals:
            relevant_indices = cumulative_timesteps[cumulative_timesteps <= step_threshold].index
            if relevant_indices.empty: continue
            last_relevant_index = relevant_indices[-1]
            window_start_index = max(0, last_relevant_index - window_size + 1)
            window_data = data.iloc[window_start_index : last_relevant_index + 1]
            if window_data.empty: continue
            try:
                avg_reward = window_data['r'].mean()
                avg_length = window_data['l'].mean()
            except Exception as e: print(f"FEHLER bei Statistikberechnung für Intervall {step_threshold}: {e}"); continue
            win_rate = np.nan; has_success_col = 'is_success' in window_data.columns; win_rate_desc = "(Nicht verfügbar)"
            if has_success_col:
                 try:
                     success_numeric = pd.to_numeric(window_data['is_success'], errors='coerce').fillna(0)
                     success_bool = success_numeric.astype(bool); win_rate = success_bool.mean() * 100
                     if not np.isnan(win_rate):
                         if win_rate > 75: win_rate_desc = "Sehr Hoch"
                         elif win_rate > 50: win_rate_desc = "Hoch"
                         elif win_rate > 25: win_rate_desc = "Mittel"
                         else: win_rate_desc = "Niedrig"
                         win_rate_desc += f" ({win_rate:.0f}%)"
                     else: win_rate_desc = "(Berechnung fehlgeschlagen)"
                 except Exception as e: print(f"WARNUNG: Konnte 'is_success' nicht zuverlässig interpretieren bei Schritt {step_threshold}: {e}"); has_success_col = False; win_rate_desc = "(Fehler bei Interpretation)"
            report_lines.append(f"* **Nach ca. {step_threshold} Schritten:**")
            reward_desc = "Eher schlecht"
            if not np.isnan(avg_reward):
                if avg_reward > 10: reward_desc = "Sehr gut"
                elif avg_reward > 5: reward_desc = "Gut"
                elif avg_reward > 0: reward_desc = "Okay / Verbessert"
                reward_desc += f" (Avg. Belohnung: {avg_reward:.2f})"
            else: reward_desc = "(Berechnung fehlgeschlagen)"
            report_lines.append(f"  * **Ergebnis:** {reward_desc}"); report_lines.append(f"  * **Gewinnrate:** {win_rate_desc}")
            length_desc = "Eher lang"
            if not np.isnan(avg_length):
                max_turns_ref = 100
                if 'rpg_config' in sys.modules and hasattr(rpg_config, 'MAX_TURNS'): max_turns_ref = rpg_config.MAX_TURNS
                if avg_length < max_turns_ref * 0.3: length_desc = "Sehr kurz"
                elif avg_length < max_turns_ref * 0.6: length_desc = "Kurz"
                elif avg_length < max_turns_ref * 0.9: length_desc = "Mittel"
                length_desc += f" (ca. {avg_length:.0f} Runden)"
            else: length_desc = "(Berechnung fehlgeschlagen)"
            report_lines.append(f"  * **Kampfdauer:** {length_desc}")
            strategy_desc = "Noch unklar."
            if not np.isnan(avg_reward) and not np.isinf(last_avg_reward):
                 if avg_reward > last_avg_reward + 1.0: strategy_desc = "Verbessert sich deutlich."
                 elif avg_reward > last_avg_reward: strategy_desc = "Verbessert sich leicht."
                 elif avg_reward < last_avg_reward - 1.0: strategy_desc = "Verschlechtert sich."
                 else: strategy_desc = "Stagniert / Stabil."
            report_lines.append(f"  * **Strategie:** {strategy_desc} *(Hinweis: Genaue Aktionsanalyse erfordert separate Daten)*"); report_lines.append("")
            last_avg_reward = avg_reward if not np.isnan(avg_reward) else last_avg_reward
            reported_intervals += 1
        if reported_intervals == 0: print("WARNUNG: Keine Berichtsintervalle erstellt."); report_lines.append("FEHLER: Keine Datenpunkte für Berichtsintervalle gefunden.")
        report_lines.append("---"); report_lines.append("**Fazit (vereinfacht):**");
        final_window_data = data.iloc[-window_size:]
        final_avg_reward = np.nan; final_win_rate = np.nan
        if not final_window_data.empty:
            final_avg_reward = final_window_data['r'].mean()
            if 'is_success' in final_window_data.columns:
                try:
                    final_success_numeric = pd.to_numeric(final_window_data['is_success'], errors='coerce').fillna(0)
                    final_success_bool = final_success_numeric.astype(bool); final_win_rate = final_success_bool.mean() * 100
                except Exception as e: print(f"WARNUNG: Konnte 'is_success' für Fazit nicht interpretieren: {e}")
        if not np.isnan(final_avg_reward) and final_avg_reward > 10 and (np.isnan(final_win_rate) or final_win_rate > 75): report_lines.append(f"Der {class_name}-Agent hat erfolgreich gelernt und die Kriterien erfüllt (Avg Reward > 10, Win Rate > 75% oder NaN).")
        elif not np.isnan(final_avg_reward) and final_avg_reward > 5: report_lines.append(f"Der {class_name}-Agent zeigt gute Fortschritte (Avg Reward > 5), hat aber die Top-Kriterien noch nicht ganz erreicht.")
        elif reported_intervals > 0: report_lines.append(f"Der {class_name}-Agent hat das Training durchlaufen, aber die Leistung stagniert oder ist noch nicht optimal (Avg Reward: {final_avg_reward:.2f}).")
        else: report_lines.append(f"Der {class_name}-Agent hat das Training abgeschlossen, aber es konnten keine aussagekräftigen Daten für ein Fazit gewonnen werden.")
        report_lines.append("---"); final_report = "\n".join(report_lines); os.makedirs(os.path.dirname(output_filepath), exist_ok=True)
        with open(output_filepath, "w", encoding="utf-8") as f: f.write(final_report); print(f"Verständlicher Bericht erfolgreich gespeichert: {output_filepath}")
    except pd.errors.EmptyDataError: print(f"FEHLER: Monitor-Datei {monitor_file_path} enthält keine Daten nach Header.")
    except FileNotFoundError: print(f"FEHLER: Monitor-Datei nicht gefunden: {monitor_file_path}")
    except KeyError as e: print(f"FEHLER: Erwartete Spalte '{e}' nicht in {monitor_file_path} gefunden oder Datenproblem.")
    except Exception as e: print(f"FEHLER beim Generieren des Berichts: {e}"); traceback.print_exc()


# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Funktion zur detaillierten Evaluierung (V26 - Redundante Zuweisungen entfernt)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

# Konstante auf Modulebene definieren (aus V24)
N_FIGHTS_PER_OPPONENT = 50

# Hilfsfunktion zum sicheren Abrufen von Modul-Infos (unverändert zu V24)
def get_module_info(module_name: str) -> str:
    # (Code unverändert)
    try:
        module = sys.modules.get(module_name)
        if not module: return "Nicht geladen"
        version = getattr(module, '__version__', None)
        timestamp_str = "N/A"; filepath = getattr(module, '__file__', None)
        if filepath and os.path.exists(filepath):
            timestamp = os.path.getmtime(filepath); mod_time = datetime.datetime.fromtimestamp(timestamp); timestamp_str = mod_time.strftime('%Y-%m-%d %H:%M:%S')
        version_comment = "N/A"
        if filepath and os.path.exists(filepath):
             try:
                 with open(filepath, 'r', encoding='utf-8') as f:
                     first_line = f.readline()
                     if '#' in first_line and 'V' in first_line.split('#')[-1]:
                          potential_version = first_line.split('#')[-1].strip()
                          if potential_version.startswith('V') and len(potential_version) > 1 and potential_version[1].isdigit(): version_comment = potential_version
             except Exception: pass # Ignore file reading errors
        info_parts = []
        if version_comment != "N/A": info_parts.append(f"{version_comment}")
        elif version: info_parts.append(f"v{version}")
        if timestamp_str != "N/A": info_parts.append(f"({timestamp_str})")
        return " ".join(info_parts) if info_parts else "Info nicht verfügbar"
    except Exception as e: return f"Fehler beim Abrufen ({e})"


def evaluate_agent_and_summarize(
        class_name: str,
        model_dir_base: str,
        summary_dir: str,
        deterministic: bool = True,
        max_turns_per_episode: int = 100,
        training_params: Optional[dict] = None,
        output_format: str = 'txt_single' # Behalte Multi-Format Option
        ) -> None:
    """
    Evaluates a trained agent, generates summary statistics, saves detailed
    step logs and summary reports into daily subfolders within summary_dir
    (except for the comparison CSV).
    V26: Removed redundant Character/Skill assignments inside try-block.
    """
    print(f"\n--- Starte Evaluierung für Klasse: {class_name} (Format: {output_format}) ---")
    aktionswahl_str = 'Festgelegt' if deterministic else 'Zufällig (mit Erkundung)'
    print(f"  Kämpfe pro Gegner: {N_FIGHTS_PER_OPPONENT}, Aktionswahl: {aktionswahl_str}, Max Züge: {max_turns_per_episode}")

    today_str = datetime.date.today().strftime('%Y-%m-%d')
    daily_summary_dir = os.path.join(summary_dir, today_str)
    os.makedirs(daily_summary_dir, exist_ok=True)
    print(f"  Tagesordner für Berichte: {daily_summary_dir}")

    model_fname_base = f"ppo_{class_name.lower()}_agent"; model_path = os.path.join(model_dir_base, f"{model_fname_base}.zip")
    if not os.path.exists(model_path): print(f"FEHLER: Modell '{model_path}' nicht gefunden."); return

    eval_env = None; model = None
    try:
        # Module sollten global verfügbar sein durch Imports am Dateianfang
        # Stellen sicher, dass sie nicht None sind (Import oben fehlgeschlagen?)
        if 'RPGEnv' not in globals() or 'Character' not in globals() or 'Skill' not in globals():
             raise NameError("Benötigte Klassen (RPGEnv, Character, Skill) nicht global verfügbar.")
        if 'rpg_definitions' not in globals() or 'rpg_config' not in globals():
             raise NameError("Benötigte Module (rpg_definitions, rpg_config) nicht global verfügbar.")

        # --> KORREKTUR V26: Redundante Zuweisungen entfernt <--
        # Character = rpg_game_logic.Character # Nicht mehr nötig, da global importiert
        # Skill = rpg_game_logic.Skill         # Nicht mehr nötig, da global importiert

        eval_env = RPGEnv( # Direkter Zugriff auf die global importierte Klasse
             char_param_definitions=rpg_definitions.CHAR_PARAMS,
             all_buffs_debuffs_names=rpg_definitions.ALL_BUFFS_DEBUFFS_NAMES,
             max_buff_duration=rpg_definitions.MAX_BUFF_DURATION,
             max_stacks=rpg_definitions.MAX_STACKS)
        if hasattr(eval_env, 'spec') and eval_env.spec is not None:
             eval_env.spec.max_episode_steps = max_turns_per_episode
        elif hasattr(eval_env, '_max_episode_steps'):
             eval_env._max_episode_steps = max_turns_per_episode

        model = PPO.load(model_path, env=eval_env); print(f"Modell geladen: {model_fname_base}.zip")
    except Exception as e:
        print(f"FEHLER beim Laden/Erstellen der Umgebung/Modell: {e}"); traceback.print_exc();
        if eval_env: eval_env.close();
        return

    # --- Rest der Funktion (Gegnerliste, Speicher, Schleifen, Statistik, Speichern) ---
    # --- bleibt unverändert zu V25 ---
    hero_classes = ["Krieger", "Magier", "Schurke", "Kleriker"]
    if not hasattr(rpg_definitions, 'CHAR_PARAMS') or not rpg_definitions.CHAR_PARAMS:
        print("FEHLER: rpg_definitions.CHAR_PARAMS ist nicht verfügbar oder leer."); eval_env.close(); return
    ENEMY_TYPES_TO_TEST = list(set(k for k in rpg_definitions.CHAR_PARAMS if k not in hero_classes))
    if not ENEMY_TYPES_TO_TEST: print("FEHLER: Keine Gegner zum Testen gefunden."); eval_env.close(); return
    opponent_types_to_test = sorted(ENEMY_TYPES_TO_TEST)
    total_episodes_planned = len(opponent_types_to_test) * N_FIGHTS_PER_OPPONENT
    print(f"  Gegner: {opponent_types_to_test} ({len(opponent_types_to_test)} Typen)")
    print(f"  Geplante Gesamtepisoden: {total_episodes_planned}")

    total_reward_list: List[float] = []; total_turns_list: List[int] = []
    skill_usage_count: Dict[str, int] = defaultdict(int); skill_type_usage_count: Dict[str, int] = defaultdict(int)
    total_reward_components: Dict[str, float] = defaultdict(float)
    win_hero_hp_ratio_list: List[float] = []; win_hero_mana_ratio_list: List[float] = []
    loss_hero_mana_ratio_list: List[float] = []; loss_hero_hp_ratio_list: List[float] = []
    total_damage_dealt_list: List[int] = []; total_damage_taken_list: List[int] = []
    total_healing_done_list: List[int] = []
    results_by_opponent: Dict[str, Dict[str, Any]] = defaultdict(lambda: {
        'wins': 0, 'losses': 0, 'timeouts': 0, 'total_episodes': 0, 'win_turns': [], 'loss_turns': [], 'timeout_turns': [],
        'win_hp_ratios': [], 'loss_hp_ratios': [], 'win_mana_ratios': [], 'loss_mana_ratios': [],
        'damage_dealt': [], 'damage_taken': [], 'healing_done': [], 'hero_skill_usage': defaultdict(int)
    })
    enemy_actions_per_opponent: Dict[str, Dict[str, int]] = defaultdict(lambda: defaultdict(int))
    pass_count_total = 0; pass_count_skill_usable = 0; pass_resource_levels: List[float] = []
    ba_count_total = 0; ba_count_skill_usable = 0; ba_resource_levels: List[float] = []

    action_map = getattr(eval_env, 'action_to_name_map', {})
    pass_idx = getattr(eval_env, '_pass_action_idx', -1)
    basic_attack_idx = getattr(eval_env, '_basic_attack_idx', -1)
    all_skills_in_env = getattr(eval_env, 'all_skill_objects_list', [])
    initialized_skills_dict = getattr(eval_env, 'initialized_skills', {})
    primary_skill_name = rpg_definitions.PRIMARY_SKILLS.get(class_name) if hasattr(rpg_definitions, 'PRIMARY_SKILLS') else None
    primary_skill_obj: Optional[Skill] = initialized_skills_dict.get(primary_skill_name) if primary_skill_name and initialized_skills_dict else None
    main_resource_name = rpg_definitions.CLASS_MAIN_RESOURCE.get(class_name) if hasattr(rpg_definitions, 'CLASS_MAIN_RESOURCE') else None
    class_skill_names = rpg_definitions.ALL_SKILLS_MAP.get(class_name, []) if hasattr(rpg_definitions, 'ALL_SKILLS_MAP') else []
    if not action_map: print("WARNUNG: action_map in Umgebung nicht gefunden!")
    if pass_idx == -1: print("WARNUNG: Pass-Aktion Index nicht gefunden!")
    if basic_attack_idx == -1: print("WARNUNG: Basis-Angriff Index nicht gefunden!")
    if not primary_skill_obj: print(f"WARNUNG: Primärskill '{primary_skill_name}' nicht gefunden oder nicht initialisiert!")
    if not main_resource_name: print(f"WARNUNG: Hauptressource für Klasse '{class_name}' nicht definiert!")

    step_log_filename = f"evaluation_steps_{class_name.lower()}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    step_log_filepath = os.path.join(daily_summary_dir, step_log_filename)
    step_log_file = None; csv_writer = None
    step_log_header = [
        "EpisodeID", "OpponentType", "Turn", "HeroHP", "HeroMaxHP", "HeroMana", "HeroMaxMana", "HeroStamina", "HeroMaxStamina", "HeroEnergy", "HeroMaxEnergy",
        "HeroBuffs", "HeroDebuffs", "EnemyHP", "EnemyMaxHP", "EnemyMana", "EnemyMaxMana", "EnemyBuffs", "EnemyDebuffs",
        "HeroActionName", "HeroActionIndex", "EnemyActionName", "StepReward", "RewardComponents"
    ]
    try:
        step_log_file = open(step_log_filepath, 'w', newline='', encoding='utf-8')
        csv_writer = csv.writer(step_log_file)
        csv_writer.writerow(step_log_header)
        print(f"  Detailliertes Schritt-Log wird geschrieben nach: {step_log_filepath}")
    except Exception as e: print(f"FEHLER beim Öffnen/Schreiben des Schritt-Log Headers: {e}"); step_log_file = None; csv_writer = None

    start_time = time.time(); print(f"  Starte Evaluierungs-Schleifen...")
    episodes_run = 0; current_episode_global = 0
    try:
        for opponent_type in opponent_types_to_test:
            opponent_results = results_by_opponent[opponent_type]
            print(f"    Starte Kämpfe gegen: {opponent_type}")
            for ep_vs_opp in range(N_FIGHTS_PER_OPPONENT):
                current_episode_global += 1; episode_id = f"{class_name}_vs_{opponent_type}_{ep_vs_opp+1}"
                try:
                     eval_env.set_fixed_class(class_name); eval_env.set_fixed_opponent(opponent_type)
                     obs, info = eval_env.reset()
                     if not hasattr(eval_env, 'hero') or not eval_env.hero or not hasattr(eval_env, 'enemy') or not eval_env.enemy:
                         raise RuntimeError("Held und/oder Gegner nicht korrekt nach reset initialisiert.")
                     episodes_run += 1
                except Exception as e: print(f"FEHLER beim env.reset() für Ep {current_episode_global} ({episode_id}): {e}"); traceback.print_exc(); continue

                terminated, truncated = False, False; episode_reward, episode_turns = 0.0, 0;
                episode_reward_components = defaultdict(float); ep_damage_dealt = 0; ep_damage_taken = 0; ep_healing_done = 0

                while not terminated and not truncated:
                    hero = eval_env.hero; enemy = eval_env.enemy
                    if not hero or not enemy: print(f"FEHLER: Held oder Gegner verschwunden in Ep {episode_id}, Turn {episode_turns+1}"); break
                    hp_before_step = hero.current_hp; enemy_hp_before_step = enemy.current_hp
                    try:
                        action_array, _ = model.predict(obs, deterministic=deterministic); action = int(action_array.item());
                        action_name = action_map.get(action, f"Unbekannt({action})")
                        skill_usage_count[action_name] += 1; opponent_results['hero_skill_usage'][action_name] += 1
                        is_primary_skill_usable = False; current_resource_percentage = -1.0
                        if primary_skill_obj and main_resource_name and hero:
                            try:
                                if hasattr(hero, 'can_use_skill') and callable(hero.can_use_skill):
                                    is_primary_skill_usable = hero.can_use_skill(primary_skill_obj)
                                else: print(f"WARNUNG: hero-Objekt hat keine Methode 'can_use_skill' in Ep {episode_id}, T {episode_turns+1}.")
                            except Exception as e_can_use: print(f"WARNUNG: Fehler beim Aufruf von hero.can_use_skill() für Skill '{primary_skill_name}' in Ep {episode_id}, T {episode_turns+1}: {e_can_use}")
                            resource_attr = f"current_{main_resource_name.lower()}"; max_resource_attr = f"max_{main_resource_name.lower()}"
                            if hasattr(hero, resource_attr) and hasattr(hero, max_resource_attr) and getattr(hero, max_resource_attr) > 0:
                                current_resource_percentage = (getattr(hero, resource_attr) / getattr(hero, max_resource_attr)) * 100
                        if action == pass_idx:
                            pass_count_total += 1;
                            if is_primary_skill_usable: pass_count_skill_usable += 1
                            if current_resource_percentage >= 0: pass_resource_levels.append(current_resource_percentage)
                        elif action == basic_attack_idx:
                            ba_count_total += 1;
                            if is_primary_skill_usable: ba_count_skill_usable += 1
                            if current_resource_percentage >= 0: ba_resource_levels.append(current_resource_percentage)

                        obs, reward, terminated, truncated, info = eval_env.step(action)
                        enemy_action = info.get('enemy_action_name', 'Unbekannt'); enemy_actions_per_opponent[opponent_type][enemy_action] += 1
                        skill_type = "Unbekannt";
                        if action == pass_idx: skill_type = "Passen"
                        elif action == basic_attack_idx: skill_type = "Basis-Angriff"
                        elif 0 <= action < len(all_skills_in_env):
                            skill_obj = all_skills_in_env[action];
                            if isinstance(skill_obj, Skill) and hasattr(skill_obj, 'effect_type'): skill_type = skill_obj.effect_type
                        skill_type_usage_count[skill_type] += 1
                        episode_reward += reward; episode_turns += 1
                        step_reward_components = info.get('reward_components', {});
                        if isinstance(step_reward_components, dict):
                            for key, value in step_reward_components.items(): episode_reward_components[key] += value
                        if hero and enemy:
                            ep_damage_dealt += max(0, enemy_hp_before_step - enemy.current_hp); step_damage_taken = max(0, hp_before_step - hero.current_hp); ep_damage_taken += step_damage_taken
                            if step_damage_taken == 0 and hero.current_hp > hp_before_step: ep_healing_done += (hero.current_hp - hp_before_step)
                        if csv_writer and hero and enemy:
                            try:
                                def get_effects_str(char_obj, type='buff'):
                                     effects_dict = getattr(char_obj, 'active_effects', {})
                                     return "; ".join([f"{name}({eff.get('duration', '?')}/{eff.get('stacks', '?')})" for name, eff in effects_dict.items() if isinstance(eff, dict)])
                                log_row = [ episode_id, opponent_type, episode_turns, hero.current_hp, hero.max_hp, getattr(hero, 'current_mana', 0), getattr(hero, 'max_mana', 0), getattr(hero, 'current_stamina', 0), getattr(hero, 'max_stamina', 0), getattr(hero, 'current_energy', 0), getattr(hero, 'max_energy', 0), get_effects_str(hero), "", enemy.current_hp, enemy.max_hp, getattr(enemy, 'current_mana', 0), getattr(enemy, 'max_mana', 0), get_effects_str(enemy), "", action_name, action, enemy_action, reward, str(step_reward_components) ]
                                csv_writer.writerow(log_row)
                            except Exception as log_e: print(f"FEHLER beim Schreiben des Schritt-Logs (Ep {episode_id}, T {episode_turns}): {log_e}")
                    except Exception as e: print(f"FEHLER in predict/step loop (Ep {episode_id}, T {episode_turns+1}): {e}"); traceback.print_exc(); terminated = True; continue
                total_reward_list.append(episode_reward); total_turns_list.append(episode_turns);
                for key, value in episode_reward_components.items(): total_reward_components[key] += value
                total_damage_dealt_list.append(ep_damage_dealt); total_damage_taken_list.append(ep_damage_taken); total_healing_done_list.append(ep_healing_done)
                opponent_results['total_episodes'] += 1; opponent_results['damage_dealt'].append(ep_damage_dealt); opponent_results['damage_taken'].append(ep_damage_taken); opponent_results['healing_done'].append(ep_healing_done);
                hero = eval_env.hero; hero_won = info.get('is_success', False) if isinstance(info, dict) else False
                hero_mana_ratio = 0.0; hero_hp_ratio = 0.0
                if hero:
                    if hasattr(hero, 'max_mana') and hero.max_mana > 0: hero_mana_ratio = getattr(hero, 'current_mana', 0) / hero.max_mana
                    if hasattr(hero, 'max_hp') and hero.max_hp > 0: hero_hp_ratio = getattr(hero, 'current_hp', 0) / hero.max_hp
                if hero_won:
                    opponent_results['wins'] += 1; opponent_results['win_turns'].append(episode_turns); win_hero_hp_ratio_list.append(hero_hp_ratio); win_hero_mana_ratio_list.append(hero_mana_ratio); opponent_results['win_hp_ratios'].append(hero_hp_ratio); opponent_results['win_mana_ratios'].append(hero_mana_ratio);
                elif truncated and not terminated:
                    opponent_results['timeouts'] += 1; opponent_results['timeout_turns'].append(episode_turns); loss_hero_hp_ratio_list.append(hero_hp_ratio); loss_hero_mana_ratio_list.append(hero_mana_ratio); opponent_results['loss_hp_ratios'].append(hero_hp_ratio); opponent_results['loss_mana_ratios'].append(hero_mana_ratio);
                elif terminated:
                    opponent_results['losses'] += 1; opponent_results['loss_turns'].append(episode_turns); loss_hero_hp_ratio_list.append(hero_hp_ratio); loss_hero_mana_ratio_list.append(hero_mana_ratio); opponent_results['loss_hp_ratios'].append(hero_hp_ratio); opponent_results['loss_mana_ratios'].append(hero_mana_ratio);
                else: print(f"WARNUNG: Episode {episode_id} endete ohne Sieg/Niederlage/Timeout/Error? Status: term={terminated}, trunc={truncated}")
            print(f"    Kämpfe gegen {opponent_type} abgeschlossen.")
    except Exception as loop_err: print(f"\n!!! FATALER FEHLER in Evaluierungs-Schleifen: {loop_err} !!!"); traceback.print_exc()
    finally:
        if step_log_file:
            try: step_log_file.close(); print(f"  Schritt-Log Datei geschlossen: {step_log_filepath}")
            except Exception as e: print(f"FEHLER beim Schließen der Schritt-Log Datei: {e}")
        if eval_env:
            try: eval_env.close(); print("  Evaluierungsumgebung geschlossen.")
            except Exception as e: print(f"FEHLER beim Schließen der Evaluierungsumgebung: {e}")

    eval_duration = time.time() - start_time
    print(f"  Evaluierung für {class_name} abgeschlossen (oder abgebrochen). Dauer: {eval_duration:.2f} Sek.")
    if episodes_run == 0: print("WARNUNG: Keine Evaluierungs-Episoden erfolgreich abgeschlossen."); return

    # --- 6. Statistiken berechnen ---
    # (Code unverändert zu V24/V25)
    print("Berechne Statistiken...")
    report_data = None
    try:
        total_wins = sum(d['wins'] for d in results_by_opponent.values()); total_losses = sum(d['losses'] for d in results_by_opponent.values()); total_timeouts = sum(d['timeouts'] for d in results_by_opponent.values());
        valid_episodes_run = max(1, episodes_run); win_rate = (total_wins / valid_episodes_run) * 100; loss_rate = (total_losses / valid_episodes_run) * 100; timeout_rate = (total_timeouts / valid_episodes_run) * 100;
        avg_reward = np.mean(total_reward_list) if total_reward_list else 0; std_reward = np.std(total_reward_list) if total_reward_list else 0; avg_turns = np.mean(total_turns_list) if total_turns_list else 0;
        avg_win_hp_perc = np.mean(win_hero_hp_ratio_list) * 100 if win_hero_hp_ratio_list else 0; avg_win_mana_perc = np.mean(win_hero_mana_ratio_list) * 100 if win_hero_mana_ratio_list else 0;
        avg_loss_hp_perc = np.mean(loss_hero_hp_ratio_list) * 100 if loss_hero_hp_ratio_list else -1;
        avg_loss_mana_perc = np.mean(loss_hero_mana_ratio_list) * 100 if loss_hero_mana_ratio_list else -1;
        avg_dmg_dealt = np.mean(total_damage_dealt_list) if total_damage_dealt_list else 0; avg_dmg_taken = np.mean(total_damage_taken_list) if total_damage_taken_list else 0; avg_healing_done = np.mean(total_healing_done_list) if total_healing_done_list else 0;
        avg_reward_components = {k: v / valid_episodes_run for k, v in total_reward_components.items()};
        total_actions = sum(skill_usage_count.values()); action_percentages = {k: (v / total_actions) * 100 for k, v in skill_usage_count.items()} if total_actions > 0 else {};
        total_types = sum(skill_type_usage_count.values()); type_percentages = {k: (v / total_types) * 100 for k, v in skill_type_usage_count.items()} if total_types > 0 else {};
        opponent_summary_stats = {}
        for opp_class, data in results_by_opponent.items():
            opp_total = data['total_episodes'];
            if opp_total > 0:
                all_turns_vs = data['win_turns'] + data['loss_turns'] + data['timeout_turns']; total_hero_actions_vs = sum(data['hero_skill_usage'].values())
                opponent_summary_stats[opp_class] = {
                    'total_episodes': opp_total, 'win_rate': (data['wins']/opp_total)*100, 'loss_rate': (data['losses']/opp_total)*100, 'timeout_rate': (data['timeouts']/opp_total)*100,
                    'avg_win_turns': np.mean(data['win_turns']) if data['win_turns'] else 0, 'avg_loss_turns': np.mean(data['loss_turns']) if data['loss_turns'] else 0, 'avg_timeout_turns': np.mean(data['timeout_turns']) if data['timeout_turns'] else 0,
                    'avg_overall_turns': np.mean(all_turns_vs) if all_turns_vs else 0, 'avg_win_hp_perc': np.mean(data['win_hp_ratios'])*100 if data['win_hp_ratios'] else 0,
                    'avg_loss_hp_perc': np.mean(data['loss_hp_ratios'])*100 if data['loss_hp_ratios'] else -1,
                    'avg_win_mana_perc': np.mean(data['win_mana_ratios'])*100 if data['win_mana_ratios'] else 0, 'avg_loss_mana_perc': np.mean(data['loss_mana_ratios'])*100 if data['loss_mana_ratios'] else -1,
                    'avg_dmg_dealt': np.mean(data['damage_dealt']) if data['damage_dealt'] else 0, 'avg_dmg_taken': np.mean(data['damage_taken']) if data['damage_taken'] else 0, 'avg_healing_done': np.mean(data['healing_done']) if data['healing_done'] else 0,
                    'hero_skill_usage': {k: (v / total_hero_actions_vs * 100) if total_hero_actions_vs > 0 else 0 for k,v in data['hero_skill_usage'].items()}
                 }
        perc_pass_total = action_percentages.get('Passen', 0); perc_ba_total = action_percentages.get('Basis-Angriff', 0)
        perc_pass_skill_usable = (pass_count_skill_usable / pass_count_total * 100) if pass_count_total > 0 else 0; avg_pass_resource = np.mean(pass_resource_levels) if pass_resource_levels else -1;
        perc_ba_skill_usable = (ba_count_skill_usable / ba_count_total * 100) if ba_count_total > 0 else 0; avg_ba_resource = np.mean(ba_resource_levels) if ba_resource_levels else -1
        print("Statistiken berechnet.")
        print("Generiere Berichtsdaten...")
        report_data = {
            "class_name": class_name, "model_path": model_path, "eval_timestamp": datetime.datetime.now(),
            "code_versions": { name: get_module_info(name) for name in ['rpg_definitions', 'rpg_env', 'rpg_game_logic', 'rpg_training_utils', 'rpg_config']},
            "training_params": training_params or {}, "eval_params": { "fights_per_opponent": N_FIGHTS_PER_OPPONENT, "deterministic": deterministic, "max_turns": max_turns_per_episode, "opponents": opponent_types_to_test },
            "overall_stats": { "episodes_run": episodes_run, "win_rate": win_rate, "loss_rate": loss_rate, "timeout_rate": timeout_rate, "avg_reward": avg_reward, "std_reward": std_reward, "avg_turns": avg_turns,
                               "avg_win_hp_perc": avg_win_hp_perc, "avg_loss_hp_perc": avg_loss_hp_perc, "avg_win_mana_perc": avg_win_mana_perc, "avg_loss_mana_perc": avg_loss_mana_perc,
                               "avg_dmg_dealt": avg_dmg_dealt, "avg_dmg_taken": avg_dmg_taken, "avg_healing_done": avg_healing_done },
            "avg_reward_components": avg_reward_components, "action_percentages": action_percentages, "type_percentages": type_percentages,
            "pass_ba_details": { "pass_perc": perc_pass_total, "pass_count": pass_count_total, "pass_skill_usable_perc": perc_pass_skill_usable, "pass_skill_usable_count": pass_count_skill_usable, "pass_avg_res_perc": avg_pass_resource,
                                 "ba_perc": perc_ba_total, "ba_count": ba_count_total, "ba_skill_usable_perc": perc_ba_skill_usable, "ba_skill_usable_count": ba_count_skill_usable, "ba_avg_res_perc": avg_ba_resource,
                                 "primary_skill": primary_skill_name, "main_resource": main_resource_name },
            "opponent_stats": opponent_summary_stats, "enemy_actions": enemy_actions_per_opponent,
            "class_skill_names": class_skill_names, "step_log_path": step_log_filepath
        }
        print("Berichtsdaten generiert.")
    except Exception as stats_err: print(f"FEHLER bei Statistikberechnung/Berichtsdatenerstellung: {stats_err}"); traceback.print_exc(); return

    # --- Speicherlogik basierend auf output_format ---
    # (Code unverändert zu V24/V25)
    if report_data:
        print(f"Versuche Bericht im Format '{output_format}' zu speichern...")
        try:
            if output_format == 'txt_single' or output_format == 'txt_append': _save_report_txt(report_data, daily_summary_dir, append=(output_format == 'txt_append'))
            elif output_format == 'csv': _save_report_csv(report_data, summary_dir)
            elif output_format == 'html': _save_report_html(report_data, daily_summary_dir)
            else: print(f"WARNUNG: Unbekanntes Ausgabeformat '{output_format}'. Speichere als 'txt_single'."); _save_report_txt(report_data, daily_summary_dir, append=False)
            print("Bericht speichern abgeschlossen.")
        except Exception as save_e: print(f"FEHLER beim Aufruf der Speicherfunktion '{output_format}': {save_e}"); traceback.print_exc()
    else: print("WARNUNG: Keine Berichtsdaten zum Speichern vorhanden aufgrund vorheriger Fehler.")
    print(f"--- Evaluierung für Klasse: {class_name} Ende ---")


# --- Hilfsfunktionen zum Speichern der Berichte ---
# (Code für _save_report_txt, _save_report_csv, _save_report_html unverändert zu V25)
# ... (Code für _save_report_txt) ...
def _save_report_txt(data: dict, target_dir: str, append: bool):
    """ Speichert den detaillierten Bericht als formatierte TXT-Datei. """
    class_name = data['class_name']; now_str = data['eval_timestamp'].strftime("%Y%m%d_%H%M%S"); date_str = data['eval_timestamp'].strftime("%Y-%m-%d");
    base_filename = f"evaluation_summary_{class_name.lower()}"; filename = f"{base_filename}_{now_str}.txt" if not append else f"{base_filename}_daily_{date_str}.txt";
    filepath = os.path.join(target_dir, filename)
    summary_lines = []; title_sep = "#"*75; section_sep = "="*60; sub_sep = "-"*50;
    summary_lines.extend([title_sep, f" Agenten-Evaluierungsbericht: {class_name} ".center(75), title_sep]); summary_lines.append(f"Bericht generiert am: {data['eval_timestamp'].strftime('%Y-%m-%d %H:%M:%S')}\n")
    summary_lines.append("**1. Kopfzeilen / Metadaten**"); summary_lines.append(f"   - Heldenklasse:         {class_name}"); summary_lines.append(f"   - Getestetes Modell:    {data['model_path']}");
    summary_lines.append("   - Code-Versionen:"); [summary_lines.append(f"     - {name}.py: {info}") for name, info in data['code_versions'].items()]
    if data['training_params']:
        summary_lines.append("   - Trainingsparameter (Auszug):"); summary_lines.append(f"     - Learning Rate:      {data['training_params'].get('learning_rate', 'N/A')}"); summary_lines.append(f"     - Batch Size:         {data['training_params'].get('batch_size', 'N/A')}"); summary_lines.append(f"     - Gamma:              {data['training_params'].get('gamma', 'N/A')}")
    summary_lines.append("   - Evaluierungsparameter:"); summary_lines.append(f"     - Kämpfe pro Gegner:  {data['eval_params']['fights_per_opponent']}"); summary_lines.append(f"     - Aktionswahl:        {'Festgelegt' if data['eval_params']['deterministic'] else 'Zufällig (mit Erkundung)'}"); summary_lines.append(f"     - Max. Runden:        {data['eval_params']['max_turns']}"); summary_lines.append(f"     - Getestete Gegner:   {', '.join(data['eval_params']['opponents'])}"); summary_lines.append("\n")
    summary_lines.extend([section_sep, " 2. Fazit / Executive Summary ".center(60), section_sep]); stats = data['overall_stats']; ap = data['action_percentages']; rc = data['avg_reward_components']; os_stats = data['opponent_stats']; pb = data['pass_ba_details']
    fazit_lines = []; fazit_lines.append(f"* Leistung: {stats['win_rate']:.1f}% Siege über {stats['episodes_run']} Kämpfe.")
    problem_flags = []; pass_perc = pb.get('pass_perc', 0); ba_perc = pb.get('ba_perc', 0)
    if pass_perc > 15: problem_flags.append(f"hohe Passen-Rate ({pass_perc:.1f}%)")
    if ba_perc > 30: problem_flags.append(f"hohe Basis-Angriff-Rate ({ba_perc:.1f}%)")
    avg_loss_mana = stats.get('avg_loss_mana_perc', -1)
    if avg_loss_mana >= 0 and avg_loss_mana < 10: problem_flags.append("oft Mana-leer bei Niederlagen")
    fehler_reward = rc.get('Skill Nutzung (Fehler)', 0)
    if fehler_reward < -0.5: problem_flags.append(f"deutliche Skill-Fehler (Reward {fehler_reward:.2f})")
    if problem_flags: fazit_lines.append(f"* Hauptproblem(e): {', '.join(problem_flags)}.")
    else: fazit_lines.append("* Strategie wirkt aktuell relativ ausbalanciert (basierend auf einfachen Metriken).")
    major_weakness = None; min_win_rate = 101
    for opp, opp_data in os_stats.items():
        if opp_data.get('total_episodes', 0) >= data['eval_params']['fights_per_opponent'] // 2 :
            if opp_data.get('win_rate', 100) < min_win_rate:
                 min_win_rate = opp_data['win_rate']; major_weakness = opp
    if major_weakness and min_win_rate < 40: fazit_lines.append(f"* Größte Schwäche: Sehr anfällig gegen {major_weakness} ({min_win_rate:.0f}% Sieg).")
    summary_lines.extend(fazit_lines); summary_lines.append("\n")
    summary_lines.extend([section_sep, " 3. Gesamtleistung des Helden ".center(60), section_sep]); summary_lines.append(f"- Erfolgsquote: {stats['win_rate']:.1f}% Siege / {stats['loss_rate']:.1f}% Niederl. / {stats['timeout_rate']:.1f}% Timeout ({stats['episodes_run']} Episoden)"); summary_lines.append(f"- Ø Gesamtbelohnung pro Kampf: {stats['avg_reward']:.2f} (StdAbw: {stats['std_reward']:.2f})")
    win_t = []; loss_t = []; timeout_t = []; [ (win_t.extend(opp_d.get('win_turns',[])), loss_t.extend(opp_d.get('loss_turns',[])), timeout_t.extend(opp_d.get('timeout_turns',[]))) for opp_d in data['opponent_stats'].values() ]
    summary_lines.append(f"- Ø Kampfdauer: {stats['avg_turns']:.1f} Runden (Sieg: {np.mean(win_t) if win_t else 0:.1f}, Niederl.: {np.mean(loss_t) if loss_t else 0:.1f}, Timeout: {np.mean(timeout_t) if timeout_t else 0:.1f})"); summary_lines.append(f"- Ø Rest-HP bei Siegen: {stats['avg_win_hp_perc']:.1f}%"); summary_lines.append(f"- Ø Rest-HP bei Niederl./Timeout: {stats['avg_loss_hp_perc']:.1f}%"); summary_lines.append(f"- Ø Rest-Mana bei Siegen: {stats['avg_win_mana_perc']:.1f}%"); summary_lines.append(f"- Ø Rest-Mana bei Niederl./Timeout: {stats['avg_loss_mana_perc']:.1f}%"); summary_lines.append(f"- Ø Verursachter Schaden pro Kampf: {stats['avg_dmg_dealt']:.1f}"); summary_lines.append(f"- Ø Erlittener Schaden pro Kampf: {stats['avg_dmg_taken']:.1f}"); summary_lines.append(f"- Ø Gewirkte Heilung pro Kampf: {stats['avg_healing_done']:.1f}"); summary_lines.append("\n")
    summary_lines.extend([section_sep, " 4. Belohnungsanalyse (Gesamt, Ø pro Kampf) ".center(60), section_sep]); reward_name_map = {'Sieg': 'Für Siege', 'Niederlage': 'Für Niederlagen','Schaden verursacht Bonus': 'Für verursachten Schaden', 'Level Up Bonus': 'Für Level Ups','Zeit Malus': 'Für lange Kämpfe (Zeit)', 'Rest-HP Bonus (Sieg)': 'Für Rest-HP bei Sieg','Skill Nutzung (Erfolg)': 'Für erfolgreiche Skill-Nutzung', 'Timeout Malus': 'Für Timeouts','Basis-Angriff Nutzung': 'Für Nutzung Basis-Angriff', 'Skill Nutzung (Fehler)': 'Für Fehlversuche (Skill)','Passen': 'Für Passen', 'Ungültige Aktion': 'Für ungültige Aktionen', 'Aktiver Buff Bonus': 'Bonus für aktive Buffs', 'Debuff Angewendet Bonus': 'Bonus für Debuff-Anwendung', 'Basis-Angriff Malus': 'Malus für Basis-Angriff'}
    sorted_rewards = sorted(rc.items(), key=lambda item: abs(item[1]), reverse=True)
    if not sorted_rewards: summary_lines.append("  (Keine Reward-Daten)")
    else:
        summary_lines.append("* Top Positive Komponenten:"); count_pos = 0;
        for key, value in sorted_rewards:
            if value > 0.01 and count_pos < 5: summary_lines.append(f"    - {reward_name_map.get(key, key)}: {value:+.2f}"); count_pos += 1
        if count_pos == 0: summary_lines.append("    (Keine nennenswert positiven)")
        summary_lines.append("* Top Negative Komponenten:"); count_neg = 0;
        for key, value in sorted(sorted_rewards, key=lambda item: item[1]):
            if value < -0.01 and count_neg < 5: summary_lines.append(f"    - {reward_name_map.get(key, key)}: {value:+.2f}"); count_neg += 1
        if count_neg == 0: summary_lines.append("    (Keine nennenswert negativen)")
    summary_lines.append("\n")
    summary_lines.extend([section_sep, " 5. Strategie-Analyse des Helden (Gesamt) ".center(60), section_sep]); summary_lines.append("* Aktionstypen-Verteilung:"); tp = data['type_percentages']
    if tp: sorted_types = sorted(tp.items(), key=lambda item: item[1], reverse=True); [summary_lines.append(f"    - {type_name.replace('_', ' ').capitalize()}: {perc:.1f}%") for type_name, perc in sorted_types if perc > 0.1]
    else: summary_lines.append("    (Keine Daten zu Aktionstypen)")
    summary_lines.append("\n* Spezifische Aktions-Verteilung (Alle Klassenskills + Andere):"); ap = data['action_percentages']; csn = data['class_skill_names']
    if ap:
        class_skills_used = {k:v for k,v in ap.items() if k in csn}; other_actions_used = {k:v for k,v in ap.items() if k not in csn}
        sorted_class_skills = sorted(class_skills_used.items(), key=lambda item: item[1], reverse=True); sorted_other_actions = sorted(other_actions_used.items(), key=lambda item: item[1], reverse=True)
        if sorted_class_skills: summary_lines.append("  **Klassenskills:**"); [summary_lines.append(f"    - {skill_name}: {perc:.1f}%") for skill_name, perc in sorted_class_skills if perc > 0.01]
        else: summary_lines.append("  (Keine definierten Klassenskills genutzt?)")
        if sorted_other_actions: summary_lines.append("  **Andere Aktionen:**"); [summary_lines.append(f"    - {action_name}: {perc:.1f}%") for action_name, perc in sorted_other_actions if perc > 0.01]
        else: summary_lines.append("  (Keine anderen Aktionen genutzt)")
    else: summary_lines.append("    (Keine Daten zu Aktionen)")
    summary_lines.append("\n"); summary_lines.extend([sub_sep, " Detail: Passen / Basis-Angriff (Held) ".center(50), sub_sep]); pb = data['pass_ba_details']
    primary_skill_name = pb.get('primary_skill', 'N/A'); main_res_name = pb.get('main_resource', 'Ressource')
    if pb.get('pass_count', 0) > 0:
        summary_lines.append(f"* Passen genutzt ({pb['pass_perc']:.1f}% / {pb['pass_count']} mal):"); summary_lines.append(f"    - Hauptskill ('{primary_skill_name}') verfügbar in: {pb['pass_skill_usable_perc']:.1f}% d. Fälle ({pb['pass_skill_usable_count']} mal)."); avg_res = pb.get('pass_avg_res_perc', -1)
        if avg_res >= 0: summary_lines.append(f"    - Ø {main_res_name.capitalize()} dabei: {avg_res:.1f}%")
    else: summary_lines.append("* Passen: Wurde nicht genutzt.")
    if pb.get('ba_count', 0) > 0:
        summary_lines.append(f"\n* Basis-Angriff genutzt ({pb['ba_perc']:.1f}% / {pb['ba_count']} mal):"); summary_lines.append(f"    - Hauptskill ('{primary_skill_name}') verfügbar in: {pb['ba_skill_usable_perc']:.1f}% d. Fälle ({pb['ba_skill_usable_count']} mal)."); avg_res = pb.get('ba_avg_res_perc', -1)
        if avg_res >= 0: summary_lines.append(f"    - Ø {main_res_name.capitalize()} dabei: {avg_res:.1f}%")
    else: summary_lines.append("\n* Basis-Angriff: Wurde nicht genutzt.")
    summary_lines.append("\n")
    summary_lines.extend([section_sep, f" 6. Leistung des Helden gegen einzelne Gegner ".center(60), section_sep]); os_stats = data['opponent_stats']
    if os_stats:
        for opp_class, stats_opp in sorted(os_stats.items()):
            if stats_opp.get('total_episodes', 0) > 0:
                summary_lines.append(f"\n* Gegen {opp_class} (Gespielt: {stats_opp['total_episodes']}):"); summary_lines.append(f"    - Ergebnis: {stats_opp['win_rate']:.1f}% Sieg / {stats_opp['loss_rate']:.1f}% Niederl. / {stats_opp['timeout_rate']:.1f}% Timeout"); summary_lines.append(f"    - Ø Dauer: {stats_opp['avg_overall_turns']:.1f} Züge (Sieg: {stats_opp['avg_win_turns']:.1f}, Niederl.: {stats_opp['avg_loss_turns']:.1f})"); summary_lines.append(f"    - Ø Rest-HP (Held): Sieg {stats_opp['avg_win_hp_perc']:.1f}% / Niederl. {stats_opp['avg_loss_hp_perc']:.1f}%"); summary_lines.append(f"    - Ø Rest-Mana (Held): Sieg {stats_opp['avg_win_mana_perc']:.1f}% / Niederl. {stats_opp['avg_loss_mana_perc']:.1f}%"); summary_lines.append(f"    - Ø Schaden (Verurs./Erlitt.): {stats_opp['avg_dmg_dealt']:.1f} / {stats_opp['avg_dmg_taken']:.1f}"); summary_lines.append(f"    - Ø Heilung (Held): {stats_opp['avg_healing_done']:.1f}"); summary_lines.append(f"    - Helden-Strategie vs {opp_class}:")
                opp_skill_usage = stats_opp.get('hero_skill_usage', {}); top_skills = sorted(opp_skill_usage.items(), key=lambda item: item[1], reverse=True)[:4]; usage_str = ", ".join([f"{name} ({perc:.0f}%)" for name, perc in top_skills if perc > 1]); summary_lines.append(f"      -> Top Aktionen: {usage_str if usage_str else 'N/A'}")
    else: summary_lines.append("  (Keine Daten pro Gegner verfügbar)")
    summary_lines.append("\n")
    summary_lines.extend([section_sep, f" 7. Verhalten der Gegner gegen {class_name} ".center(60), section_sep]); ea = data['enemy_actions']
    if ea:
        for opp_type, action_counts in sorted(ea.items()):
            summary_lines.append(f"\n* Aktionen von {opp_type}:"); total_opp_actions = sum(action_counts.values())
            if total_opp_actions > 0:
                sorted_opp_actions = sorted(action_counts.items(), key=lambda item: item[1], reverse=True); action_str = ", ".join([f"{name} ({((count/total_opp_actions)*100):.0f}%)" for name, count in sorted_opp_actions[:4] if count > 0]); summary_lines.append(f"    -> Hauptaktionen: {action_str if action_str else 'N/A'}")
            else: summary_lines.append("    (Keine Aktionen aufgezeichnet)")
    else: summary_lines.append("  (Keine Daten zum Gegnerverhalten verfügbar)")
    summary_lines.append("\n")
    summary_lines.extend([section_sep, " 8. Anhänge / Rohdaten-Verweise ".center(60), section_sep])
    monitor_log_path = "Nicht ermittelbar"
    if 'rpg_config' in sys.modules and hasattr(rpg_config, 'LOG_DIR_BASE'): monitor_dir = os.path.join(rpg_config.LOG_DIR_BASE, f"{class_name.lower()}_monitor_logs"); monitor_file = os.path.join(monitor_dir, "monitor.csv"); monitor_log_path = monitor_file
    summary_lines.append(f"- Monitor-Log (Training): {monitor_log_path}"); summary_lines.append(f"- Detailliertes Schritt-Log (Evaluierung): {data['step_log_path']}"); summary_lines.append("\n")
    summary_lines.append(f"{title_sep}")
    file_mode = "a" if append else "w"
    try:
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        with open(filepath, file_mode, encoding="utf-8") as f:
            if append and os.path.exists(filepath) and os.path.getsize(filepath) > 0: f.write("\n\n" + "="*80 + f"\n=== Neuer Bericht angehängt am: {data['eval_timestamp'].strftime('%Y-%m-%d %H:%M:%S')} ===\n" + "="*80 + "\n\n")
            f.write("\n".join(summary_lines))
        print(f"  TXT Zusammenfassung {'angehängt an' if append else 'gespeichert in'}: {filepath}")
    except Exception as e: print(f"FEHLER beim Speichern der TXT Zusammenfassung {filepath}: {e}")

# ... (Code für _save_report_csv unverändert) ...
def _save_report_csv(data: dict, summary_dir: str):
    """ Erstellt/Aktualisiert eine CSV-Datei für den Vergleich von Läufen. """
    class_name = data['class_name']; run_timestamp = data['eval_timestamp'].strftime("%Y%m%d_%H%M%S");
    filepath = os.path.join(summary_dir, f"evaluation_comparison_{class_name.lower()}.csv"); print(f"Erstelle/Aktualisiere Vergleichs-CSV: {filepath}")
    metrics_to_save = {}; stats = data['overall_stats']; pb = data['pass_ba_details']
    metrics_to_save["Timestamp"] = run_timestamp
    metrics_to_save["Gewinnrate Gesamt (%)"] = f"{stats.get('win_rate', 0):.1f}"; metrics_to_save["Ø Belohnung"] = f"{stats.get('avg_reward', 0):.2f}"; metrics_to_save["Ø Kampfdauer"] = f"{stats.get('avg_turns', 0):.1f}"; metrics_to_save["Ø Rest-Mana (Sieg %)"] = f"{stats.get('avg_win_mana_perc', 0):.1f}"; metrics_to_save["Ø Fehler-Reward"] = f"{data.get('avg_reward_components', {}).get('Skill Nutzung (Fehler)', 0):.2f}"; metrics_to_save["Passen (%)"] = f"{pb.get('pass_perc', 0):.1f}"; metrics_to_save["Basis-Angriff (%)"] = f"{pb.get('ba_perc', 0):.1f}";
    for opp, opp_stats in data.get('opponent_stats', {}).items(): metrics_to_save[f"Gewinnrate vs {opp} (%)"] = f"{opp_stats.get('win_rate', 0):.1f}"
    try:
        if os.path.exists(filepath):
            df = pd.read_csv(filepath)
            if "Timestamp" not in df.columns: df.insert(0, "Timestamp", "")
        else: df = pd.DataFrame(columns=list(metrics_to_save.keys()))
        if run_timestamp in df["Timestamp"].values:
            print(f"WARNUNG: Zeitstempel {run_timestamp} existiert bereits in CSV. Überschreibe Zeile.")
            idx = df[df["Timestamp"] == run_timestamp].index; df.loc[idx, metrics_to_save.keys()] = metrics_to_save.values()
        else: new_row = pd.DataFrame([metrics_to_save]); df = pd.concat([df, new_row], ignore_index=True)
        try:
             df['Timestamp_dt'] = pd.to_datetime(df['Timestamp'], format='%Y%m%d_%H%M%S')
             df = df.sort_values(by='Timestamp_dt', ascending=False).drop(columns=['Timestamp_dt'])
        except ValueError: print("WARNUNG: Konnte Timestamp-Spalte nicht für Sortierung konvertieren.")
        df.to_csv(filepath, index=False)
        print(f"  Vergleichs-CSV erfolgreich gespeichert: {filepath}")
    except Exception as e: print(f"FEHLER beim Speichern/Aktualisieren der Vergleichs-CSV {filepath}: {e}"); traceback.print_exc()

# ... (Code für _save_report_html unverändert) ...
def _save_report_html(data: dict, target_dir: str):
    """ Speichert den Bericht als HTML-Datei. """
    class_name = data['class_name']; now_str = data['eval_timestamp'].strftime("%Y%m%d_%H%M%S");
    filename = f"evaluation_summary_{class_name.lower()}_{now_str}.html"; filepath = os.path.join(target_dir, filename); print(f"Erstelle HTML-Bericht: {filepath}")
    def escape(text): return html.escape(str(text))
    code_versions_rows = ''.join(f"<tr><td>{escape(name)}.py</td><td>{escape(info)}</td></tr>" for name, info in data.get('code_versions', {}).items())
    opponent_rows = ''
    for opp, stats_opp in sorted(data.get('opponent_stats', {}).items()):
         top_skills_list = sorted(stats_opp.get('hero_skill_usage', {}).items(), key=lambda item: item[1], reverse=True)[:3]
         top_skills_str = ", ".join([f"{escape(n)} ({p:.0f}%)" for n, p in top_skills_list if p > 1]) or 'N/A'
         opponent_rows += f"""<tr><td>{escape(opp)}</td><td>{stats_opp.get('win_rate', 0):.1f}</td><td>{stats_opp.get('avg_overall_turns', 0):.1f}</td><td>{stats_opp.get('avg_win_hp_perc', 0):.1f}</td><td>{stats_opp.get('avg_loss_hp_perc', -1):.1f}</td><td>{stats_opp.get('avg_win_mana_perc', 0):.1f}</td><td>{stats_opp.get('avg_loss_mana_perc', -1):.1f}</td><td>{top_skills_str}</td></tr>"""
    action_type_list = ''.join(f"<li>{escape(type_name.replace('_', ' ').capitalize())}: {perc:.1f}%</li>" for type_name, perc in sorted(data.get('type_percentages', {}).items(), key=lambda item: item[1], reverse=True) if perc > 0.1)
    ap = data.get('action_percentages', {}); csn = data.get('class_skill_names', [])
    class_skills_used_html = ""; other_actions_used_html = ""
    if ap:
        class_skills_used = {k:v for k,v in ap.items() if k in csn}; other_actions_used = {k:v for k,v in ap.items() if k not in csn}
        sorted_class_skills = sorted(class_skills_used.items(), key=lambda item: item[1], reverse=True); sorted_other_actions = sorted(other_actions_used.items(), key=lambda item: item[1], reverse=True)
        if sorted_class_skills: class_skills_used_html = "<h5>Klassenskills</h5><ul>" + ''.join(f"<li>{escape(skill_name)}: {perc:.1f}%</li>" for skill_name, perc in sorted_class_skills if perc > 0.01) + "</ul>"
        if sorted_other_actions: other_actions_used_html = "<h5>Andere Aktionen</h5><ul>" + ''.join(f"<li>{escape(action_name)}: {perc:.1f}%</li>" for action_name, perc in sorted_other_actions if perc > 0.01) + "</ul>"
    specific_action_list = class_skills_used_html + other_actions_used_html
    pb = data.get('pass_ba_details', {})
    pass_ba_table = f"""<h4>Passen / Basis-Angriff Details</h4><table>
          <tr><th>Aktion</th><th>Anteil (%)</th><th>Anzahl</th><th>Hauptskill verfügbar (%)</th><th>Hauptskill verfügbar (Anzahl)</th><th>Ø {escape(pb.get('main_resource', 'Ressource'))} (%)</th></tr>
          <tr><td>Passen</td><td>{pb.get('pass_perc', 0):.1f}</td><td>{pb.get('pass_count', 0)}</td><td>{pb.get('pass_skill_usable_perc', 0):.1f}</td><td>{pb.get('pass_skill_usable_count', 0)}</td><td>{pb.get('pass_avg_res_perc', -1):.1f if pb.get('pass_avg_res_perc', -1) >= 0 else 'N/A'}</td></tr>
          <tr><td>Basis-Angriff</td><td>{pb.get('ba_perc', 0):.1f}</td><td>{pb.get('ba_count', 0)}</td><td>{pb.get('ba_skill_usable_perc', 0):.1f}</td><td>{pb.get('ba_skill_usable_count', 0)}</td><td>{pb.get('ba_avg_res_perc', -1):.1f if pb.get('ba_avg_res_perc', -1) >= 0 else 'N/A'}</td></tr>
        </table><p><small>Hauptskill: {escape(pb.get('primary_skill', 'N/A'))}</small></p>"""
    stats = data.get('overall_stats', {})
    html_content = f"""<!DOCTYPE html><html lang="de"><head><meta charset="UTF-8"><title>Evaluierungsbericht: {escape(class_name)}</title><style>body{{font-family: sans-serif; margin: 20px; line-height: 1.6;}} table{{border-collapse: collapse; margin-bottom: 20px; width: auto;}} th, td{{border: 1px solid #ccc; padding: 8px; text-align: left; vertical-align: top;}} th{{background-color: #f2f2f2; font-weight: bold;}} h1, h2, h3 {{border-bottom: 1px solid #ddd; padding-bottom: 5px; margin-top: 30px;}} h1 {{font-size: 1.8em;}} h2 {{font-size: 1.5em;}} h3 {{font-size: 1.2em;}} .code{{font-family: monospace; background-color: #eee; padding: 2px 4px; border: 1px solid #ddd; border-radius: 3px; word-wrap: break-word;}} .metrics-table td:first-child{{font-weight: bold;}} ul {{margin-top: 5px; padding-left: 20px;}} li {{margin-bottom: 5px;}} details > summary {{ cursor: pointer; font-weight: bold; margin-bottom: 10px; }}</style></head><body> <h1>Evaluierungsbericht: {escape(class_name)}</h1> <p>Bericht generiert am: {data['eval_timestamp'].strftime('%Y-%m-%d %H:%M:%S')}</p>
<details open><summary><h2>1. Metadaten & Setup</h2></summary> <table> <tr><th>Parameter</th><th>Wert</th></tr> <tr><td>Getestetes Modell</td><td><span class="code">{escape(data.get('model_path', 'N/A'))}</span></td></tr> <tr><td>Kämpfe pro Gegner</td><td>{escape(data.get('eval_params', {}).get('fights_per_opponent', 'N/A'))}</td></tr> <tr><td>Aktionswahl</td><td>{'Festgelegt (Deterministic)' if data.get('eval_params', {}).get('deterministic', True) else 'Zufällig (Stochastic)'}</td></tr> <tr><td>Max. Runden pro Kampf</td><td>{escape(data.get('eval_params', {}).get('max_turns', 'N/A'))}</td></tr> <tr><td>Gegner</td><td>{escape(", ".join(data.get('eval_params', {}).get('opponents', [])))}</td></tr> </table> <h3>Code-Versionen</h3> <table><tr><th>Datei</th><th>Info</th></tr>{code_versions_rows}</table> </details>
<details open><summary><h2>2. Gesamtleistung ({escape(stats.get('episodes_run', 0))} Kämpfe)</h2></summary> <table class="metrics-table"> <tr><th>Metrik</th><th>Wert</th></tr> <tr><td>Gewinnrate</td><td>{stats.get('win_rate', 0):.1f}%</td></tr> <tr><td>Verlustrate</td><td>{stats.get('loss_rate', 0):.1f}%</td></tr> <tr><td>Timeout-Rate</td><td>{stats.get('timeout_rate', 0):.1f}%</td></tr> <tr><td>Ø Belohnung</td><td>{stats.get('avg_reward', 0):.2f} (±{stats.get('std_reward', 0):.2f})</td></tr> <tr><td>Ø Kampfdauer</td><td>{stats.get('avg_turns', 0):.1f} Runden</td></tr> <tr><td>Ø Rest-HP (Sieg)</td><td>{stats.get('avg_win_hp_perc', 0):.1f}%</td></tr> <tr><td>Ø Rest-HP (Niederl./Timeout)</td><td>{stats.get('avg_loss_hp_perc', -1):.1f}%</td></tr> <tr><td>Ø Rest-Mana (Sieg)</td><td>{stats.get('avg_win_mana_perc', 0):.1f}%</td></tr> <tr><td>Ø Rest-Mana (Niederl./Timeout)</td><td>{stats.get('avg_loss_mana_perc', -1):.1f}%</td></tr> <tr><td>Ø Schaden Verurs.</td><td>{stats.get('avg_dmg_dealt', 0):.1f}</td></tr> <tr><td>Ø Schaden Erlitten</td><td>{stats.get('avg_dmg_taken', 0):.1f}</td></tr> <tr><td>Ø Heilung Gewirkt</td><td>{stats.get('avg_healing_done', 0):.1f}</td></tr> </table> </details>
<details><summary><h2>3. Leistung gegen einzelne Gegner</h2></summary> <table> <tr><th>Gegner</th><th>Win Rate (%)</th><th>Ø Dauer</th><th>Ø Rest-HP Sieg (%)</th><th>Ø Rest-HP Niederl. (%)</th><th>Ø Rest-Mana Sieg (%)</th><th>Ø Rest-Mana Niederl. (%)</th><th>Top Aktionen Held</th></tr> {opponent_rows} </table> </details>
<details><summary><h2>4. Strategie-Analyse (Gesamt)</h2></summary> <h3>Aktionstypen</h3> <ul>{action_type_list if action_type_list else "<li>(Keine Daten)</li>"}</ul> <h3>Spezifische Aktionen</h3> {specific_action_list if specific_action_list else "<p>(Keine Daten)</p>"} {pass_ba_table} </details>
<details><summary><h2>5. Rohdaten-Verweise</h2></summary> <p>Monitor-Log (Training): <span class="code">{escape(data.get('monitor_log_path', 'N/A'))}</span></p> <p>Detailliertes Schritt-Log (Evaluierung): <span class="code">{escape(data.get('step_log_path', 'N/A'))}</span></p> </details>
</body></html>"""
    try:
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        with open(filepath, "w", encoding="utf-8") as f: f.write(html_content)
        print(f"  HTML Zusammenfassung gespeichert: {filepath}")
    except Exception as e: print(f"FEHLER beim Speichern der HTML Zusammenfassung {filepath}: {e}")

# --- Ende Hilfsfunktionen ---

print("Trainings- und Evaluierungs-Utilities in rpg_training_utils.py geschrieben/überschrieben (V26 - Redundante Zuweisungen entfernt).")

Overwriting rpg_training_utils.py
